# Chess · Fresh 30M training
Select a **GPU runtime**, then **Runtime → Run all**. This notebook starts a new **30,017,409-parameter** dense network (**784 → 6400 → 3904 → 1**) using the complete **full_pipeline** workflow. **SOURCE_RUN is blank** and no saved classical prior is imported. Hidden weights initialize randomly; the output head starts at zero over the built-in material/PST baseline. No previously trained model is used.

The new experiment is **fresh_30m_v1** under **MyDrive/chess_30m/**, separate from older models. The session budget is **2.5 hours** and the dense population has **32 members**. Run all again with the same settings to resume this new model, rather than reinitializing it each session.

The workflow performs **search-based dense evolution → classical genetic parameter training → optional sparse labeled training → independent evaluator/search matches → tactical/endgame diagnostics → champion export**. The sparse evaluator also starts fresh and learns from public position labels. Candidates need independent match evidence before promotion. Stages can span several sessions; more parameters do not establish stronger play.

Code is collapsed into runnable forms. No API starts here. Original Drive folders are left untouched. To intentionally start another independent experiment later, choose a new RUN_NAME.


In [1]:
#@title 1 · Training settings { display-mode: "form" }
import os, sys, platform, random
# Set before NumPy and worker imports to prevent each process spawning a BLAS thread pool.
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ[name]='1'
from pathlib import Path
def usable_cpu_count():
    cores = os.cpu_count() or 1
    if hasattr(os, 'sched_getaffinity'):
        try: cores = min(cores, len(os.sched_getaffinity(0)))
        except OSError: pass
    try:
        quota, period = Path('/sys/fs/cgroup/cpu.max').read_text().split()
        if quota != 'max': cores = min(cores, max(1, int(quota)//int(period)))
    except (OSError, ValueError):
        try:
            quota = int(Path('/sys/fs/cgroup/cpu/cpu.cfs_quota_us').read_text())
            period = int(Path('/sys/fs/cgroup/cpu/cpu.cfs_period_us').read_text())
            if quota > 0: cores = min(cores, max(1, quota//period))
        except (OSError, ValueError): pass
    return max(1, cores)

def available_ram_bytes():
    available = None
    try:
        rows = dict(line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines())
        available = int(rows['MemAvailable'].split()[0])*1024
    except (OSError, KeyError, ValueError): pass
    # A container may have less RAM than the host's /proc/meminfo reports.
    for limit_path, used_path in (
        ('/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory.current'),
        ('/sys/fs/cgroup/memory/memory.limit_in_bytes', '/sys/fs/cgroup/memory/memory.usage_in_bytes')):
        try:
            limit = Path(limit_path).read_text().strip()
            if limit != 'max':
                remaining = max(0, int(limit)-int(Path(used_path).read_text()))
                available = remaining if available is None else min(available, remaining)
        except (OSError, ValueError): pass
    return available

def automatic_workers():
    cores = usable_cpu_count()
    available = available_ram_bytes()
    # Budget estimate: reserve 1.5 GiB for the notebook, then 512 MiB per worker.
    # Actual use varies with positions, search limits and optional stage 2.
    if available is not None:
        cores = min(cores, max(1, int((available-1.5*2**30)//(512*2**20))))
    return max(1, cores)

def print_resources():
    import subprocess
    available = available_ram_bytes()
    ram = f'{available/2**30:.1f} GiB available' if available is not None else 'availability not detected'
    print(f"Resources | {usable_cpu_count()} available CPU cores | {CFG['workers']} workers | RAM: {ram}")
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total,memory.used,utilization.gpu', '--format=csv,noheader'],
            capture_output=True, text=True, check=True, timeout=3)
        print('GPU (name, total VRAM MiB, used MiB, utilization %):', result.stdout.strip())
    except (OSError, subprocess.SubprocessError):
        print('GPU: no NVIDIA device detected; CPU evolution remains available.')
    print('GPU neuroevolution uses CUDA in the training step; classic mode remains CPU-based.')

TRAINING_MODE = 'full_pipeline' #@param ['full_pipeline', 'gpu_neuroevolution', 'classic', 'parameter_evolution']
TRAINING_HOURS = 2.5 #@param {type:'number'}
RUN_NAME = '' #@param {type:'string'}
SOURCE_RUN = '' #@param {type:'string'}
TRAIN_SPARSE_EVALUATOR = True #@param {type:'boolean'}
# Fresh 30M training. Blank SOURCE_RUN and gpu_seed_classic_run import no learned weights. Reuse RUN_NAME to resume this new experiment.
QUICK_TEST = False  # Full evolution with the configured TRAINING_HOURS budget.
CFG = {
    'QUICK_TEST': QUICK_TEST, 'training_mode': TRAINING_MODE,
    'root': '/content/drive/MyDrive/chess_30m',
    'run_name': ('gpu_smoke_v1' if QUICK_TEST else 'gpu_es_30m_v1') if TRAINING_MODE=='gpu_neuroevolution' else ('quick_v1' if QUICK_TEST else 'full_v1'),
    'seed': 42, 'workers': automatic_workers(),
    'pop_size': 12 if QUICK_TEST else 64, 'generations': 5 if QUICK_TEST else 100,
    'depth': 2 if QUICK_TEST else 3, 'move_seconds': .01 if QUICK_TEST else .08,
    'node_limit': 64 if QUICK_TEST else 6000, 'max_plies': 80 if QUICK_TEST else 400,
    'pairs_per_opponent': 1 if QUICK_TEST else 3, 'peer_opponents': 1 if QUICK_TEST else 3,
    'elite': 2 if QUICK_TEST else 6, 'tournament': 3, 'blx_alpha': .25,
    'mutation_sigma': .25, 'mutation_rate': .12, 'diversity_floor': .08, 'novelty_weight': .02,
    'hall_size': 8, 'anchor_every': 5, 'anchor_elo': 1350,
    'stockfish': '/usr/games/stockfish', 'stockfish_seconds': .002 if QUICK_TEST else .03,
    'fast_win_bonus': .01, 'fault_penalty': .1, 'forfeit_seconds': 10.,
    'max_hours': .15 if QUICK_TEST else TRAINING_HOURS,
    'eval_games': 20, 'eval_levels': [1350,1600,1800,2000],
    'eval_difficulties': ['easy','medium','hard'], 'eval_max_hours': .1 if QUICK_TEST else 8.,
    'run_evaluation': QUICK_TEST and TRAINING_MODE=='classic',  # Full-run benchmarks are opt-in after training.
    'cpp_backend': not QUICK_TEST and TRAINING_MODE=='classic',  # Compile fast search for new full runs.
    'gpu_require_cuda': True,
    'gpu_search_backend': 'alpha_beta',  # 'beam' retains the earlier search for comparisons.
    'gpu_search_options': {},  # Selective pruning stays opt-in until match validation.
    'gpu_validation_pairs': 32, 'gpu_validation_seconds': .03, 'gpu_validation_nodes': 256,
    'gpu_population': 8 if QUICK_TEST else 32,  # Mirrored pairs; smaller population offsets the larger network's VRAM use.
    'gpu_hidden': [64,32] if QUICK_TEST else [6400,3904],
    'gpu_parallel_games': 64 if QUICK_TEST else min(2048,max(256,automatic_workers()*256)),
    'gpu_batch_rows': 4096, 'gpu_cpu_chunk': 16,
    'gpu_target_vram_gb': 10.0, 'gpu_headroom_gb': 3.0,
    'gpu_native_prepare': True, 'gpu_native_path': '',
    'gpu_pairs_per_opponent': 1 if QUICK_TEST else 2,
    'gpu_generations': 5 if QUICK_TEST else 10000,
    'gpu_sigma': .1, 'gpu_learning_rate': .03, 'gpu_residual_cp': 600.,
    'gpu_temperature': 30., 'gpu_max_plies': 80 if QUICK_TEST else 256,
    'gpu_hall_size': 4, 'gpu_seed_classic_run': '',
    'gpu_checkpoint_seconds': 60., 'gpu_progress_seconds': 30.,
    'gpu_anchor_enabled': True, 'gpu_anchor_every': 5,
    'STAGE2': False, 'stage2_positions': 2048, 'stage2_warm_steps': 300,
    'stage2_iterations': 10, 'stage2_pairs': 4, 'stage2_sigma': .02, 'stage2_lr': .01,
    'quick_play': False,
    'difficulty': {
        'easy': {'depth':2,'seconds':.15,'nodes':2000,'temperature':120.,'inaccuracy':.12},
        'medium': {'depth':3,'seconds':.5,'nodes':30000,'temperature':0.,'inaccuracy':0.},
        'hard': {'depth':64,'seconds':2.5,'nodes':180000,'temperature':0.,'inaccuracy':0.}}
}
CFG.update(pipeline_source_run=SOURCE_RUN.strip(), pipeline_sparse=TRAIN_SPARSE_EVALUATOR,
           pipeline_move_seconds=.25, pipeline_nodes=10000, pipeline_validation_pairs=32,
           pipeline_label_positions=20000, pipeline_sparse_epochs=100, pipeline_labels='')
if TRAINING_MODE=='full_pipeline':
    CFG['run_name']='fresh_30m_smoke_v1' if QUICK_TEST else 'fresh_30m_v1'
if TRAINING_MODE=='parameter_evolution':
    CFG['run_name']='parameters_smoke_v1' if QUICK_TEST else 'parameters_v1'
if RUN_NAME.strip():
    CFG['run_name'] = RUN_NAME.strip()
assert TRAINING_HOURS > 0
assert CFG['gpu_population']>=2 and CFG['gpu_population']%2==0
assert CFG['pop_size']>=4 and 0<CFG['elite']<CFG['pop_size']
assert CFG['eval_games']>=20 and CFG['eval_games']%2==0
assert CFG['peer_opponents']<CFG['pop_size']
random.seed(CFG['seed'])
print(platform.platform(), '| Python',sys.version.split()[0])
print_resources()
print('Mode:',CFG['training_mode'],'|', 'QUICK_TEST' if QUICK_TEST else 'FULL TRAINING')
print('Session budget:',CFG['max_hours'],'hours. Run all again to resume this run.')

if TRAINING_MODE in ('gpu_neuroevolution','full_pipeline'):
    network_sizes = [784, *CFG['gpu_hidden'], 1]
    network_parameters = sum((a + 1) * b for a, b in zip(network_sizes, network_sizes[1:]))
    print('Network:', ' → '.join(map(str, network_sizes)), '|', f'{network_parameters:,}', 'parameters')


Linux-6.6.159+-x86_64-with-glibc2.39 | Python 3.13.16
Resources | 2 available CPU cores | 2 workers | RAM: 11.7 GiB available
GPU (name, total VRAM MiB, used MiB, utilization %): Tesla T4, 15360 MiB, 0 MiB, 0 %
GPU neuroevolution uses CUDA in the training step; classic mode remains CPU-based.
Mode: full_pipeline | FULL TRAINING
Session budget: 2.5 hours. Run all again to resume this run.
Network: 784 → 6400 → 3904 → 1 | 30,017,409 parameters


In [2]:
#@title 2 · Connect Drive and prepare the runtime { display-mode: "form" }
import subprocess, shutil, json, importlib, time, hashlib
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
ROOT=Path(CFG['root'])
if not str(ROOT).startswith('/content/drive/MyDrive/'):
    raise ValueError('Choose a project folder under /content/drive/MyDrive/')
for name in ('checkpoints','models','logs','pgn'):
    (ROOT/name).mkdir(parents=True,exist_ok=True)
MODELS=ROOT/'models'
os.environ['MPLCONFIGDIR']=str(ROOT/'logs'/'matplotlib-cache')
subprocess.check_call([sys.executable,'-m','pip','install','--quiet','python-chess==1.999','numpy>=1.24,<3','tqdm>=4.66','matplotlib>=3.8'])
if CFG['training_mode'] in ('gpu_neuroevolution','full_pipeline'):
    if importlib.util.find_spec('torch') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','--quiet','torch>=2.4'])
    import torch
    if CFG['gpu_require_cuda'] and not torch.cuda.is_available():
        raise RuntimeError('Select a GPU runtime in Colab, reconnect, and Run all again. This notebook will not silently train on CPU.')
    if torch.cuda.is_available(): print('CUDA ready:',torch.cuda.get_device_name(0), '| PyTorch',torch.__version__)
# Numba is optional; the reference implementation does not depend on it.
subprocess.check_call(['apt-get','update','-qq'])
subprocess.check_call(['apt-get','install','-y','-qq','stockfish','g++'])
CFG['stockfish']=shutil.which('stockfish') or '/usr/games/stockfish'
import numpy as np
import chess, chess.engine
np.random.seed(CFG['seed'])
with chess.engine.SimpleEngine.popen_uci(CFG['stockfish']) as sf:
    option=sf.options['UCI_Elo']
    CFG['anchor_elo']=max(option.min,min(option.max,CFG['anchor_elo']))
    print('Stockfish:',sf.id,'| supported Elo:',option.min,'–',option.max)
(ROOT/'logs'/'environment.json').write_text(json.dumps({'python':sys.version,'platform':platform.platform(),'cpu_count':os.cpu_count(),'numpy':np.__version__,'chess':chess.__version__},indent=2))
print('Drive artifacts:',ROOT)

if CFG['training_mode']=='full_pipeline':
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','requests','zstandard'])


Mounted at /content/drive
CUDA ready: Tesla T4 | PyTorch 2.11.0+cu130
Stockfish: {'name': 'Stockfish 16', 'author': 'the Stockfish developers (see AUTHORS file)'} | supported Elo: 1320 – 3190
Drive artifacts: /content/drive/MyDrive/chess_30m


In [3]:
#@title 3 · Prepare the engine and verify its rules { display-mode: "form" }
CORE_SOURCE = r'''
from __future__ import annotations
import json, math, time, os, random, hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import chess, chess.polyglot, chess.pgn, chess.engine

VERSION = 'cristopher-evo-1'
MATE = 30000
JS_OPENING_BOOK = {}  # The actual JavaScript has no stored opening moves.

class JSBotPlayer:
    """Same uniform legal-move distribution as js/cristopher.js, without UI delay."""
    def __init__(self, seed=42, simulate_delay=False):
        self.rng = random.Random(seed)
        self.simulate_delay = simulate_delay
    def get_move(self, board):
        # JS constructs a fresh board from FEN, so earlier repetition history is unavailable.
        position = chess.Board(board.fen())
        if position.is_game_over() or position.is_fifty_moves():
            raise ValueError('This position has no available move.')
        if self.simulate_delay:
            time.sleep(self.rng.uniform(.4, .8))
        return self.rng.choice(list(position.legal_moves))

def fen_prefix(board):
    return ' '.join(board.fen().split()[:4])

def board_from_history(fen, history=None):
    target = chess.Board(fen)
    if not target.is_valid():
        raise ValueError('Invalid chess position')
    if history:
        replay = chess.Board()
        try:
            for uci in history:
                replay.push_uci(uci)
        except ValueError as error:
            raise ValueError('History must contain legal UCI moves from the initial position') from error
        if replay.fen() != target.fen():
            raise ValueError('History does not reach the supplied FEN')
        return replay
    return target

def random_opening(seed, min_plies=4, max_plies=6):
    rng = random.Random(seed)
    board = chess.Board()
    for _ in range(rng.randint(min_plies, max_plies)):
        moves = list(board.legal_moves)
        rng.shuffle(moves)
        for move in moves:
            board.push(move)
            if not board.is_game_over(claim_draw=True):
                break
            board.pop()
        else:
            break
    return [m.uci() for m in board.move_stack]

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    with temp.open('w') as handle:
        json.dump(value, handle, allow_nan=False)
        handle.flush(); os.fsync(handle.fileno())
    os.replace(temp, path)

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def publish_champion(cfg, artifact, engine_source, weights=None):
    """Publish a complete, immutable engine bundle before replacing its pointer."""
    import uuid
    folder = Path(cfg['root']) / 'models'
    folder.mkdir(parents=True, exist_ok=True)
    identity = {key: artifact.get(key) for key in
                ('version', 'genome', 'stage2_accepted', 'difficulty', 'opening_book')}
    identity['champion_generation'] = artifact.get('champion_generation', artifact.get('generation'))
    identity['run_name'] = cfg['run_name']
    identity['source'] = hashlib.sha256(engine_source.encode()).hexdigest()
    if weights is not None:
        identity['weights'] = hashlib.sha256(weights['weights'].detach().cpu().numpy().tobytes()).hexdigest()
        identity['prior'] = digest(weights['prior'])
    fingerprint = digest(identity)
    pointer = folder / 'latest_champion.json'
    try:
        previous = json.loads(pointer.read_text())
        previous_folder = folder / previous['folder']
        if previous['fingerprint'] == fingerprint and all(
                (previous_folder / name).is_file() and
                hashlib.sha256((previous_folder / name).read_bytes()).hexdigest() == checksum
                for name, checksum in previous['sha256'].items()):
            return previous_folder
    except (OSError, ValueError, KeyError):
        pass
    snapshot = folder / 'deployments' / (fingerprint[:16] + '-' + uuid.uuid4().hex[:8])
    snapshot.mkdir(parents=True, exist_ok=False)
    # No reader can discover this bundle until every file has been fully written.
    with (snapshot / 'cristopher_engine.py').open('w') as handle:
        handle.write(engine_source)
        handle.flush(); os.fsync(handle.fileno())
    artifact = dict(artifact)
    if weights is not None:
        import torch
        artifact['weights'] = 'weights.pt'
        with (snapshot / 'weights.pt').open('wb') as handle:
            torch.save(weights, handle)
            handle.flush(); os.fsync(handle.fileno())
    atomic_json(snapshot / 'cristopher_genome.json', artifact)
    checksums = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in snapshot.iterdir()}
    manifest = {'folder': str(snapshot.relative_to(folder)), 'fingerprint': fingerprint,
                'run_name': cfg['run_name'], 'version': artifact['version'],
                'champion_generation': identity['champion_generation'], 'sha256': checksums}
    atomic_json(snapshot / 'manifest.json', manifest)
    atomic_json(pointer, manifest)  # Commit point: the old pointer survives any earlier failure.
    print(f"Champion saved to Drive | generation {identity['champion_generation']} | {snapshot.name}", flush=True)
    return snapshot

def publish_classic_champion(cfg, state, engine_source, accepted=None, report=None):
    champion = state.get('champion')
    if not champion:
        return None
    artifact = {'version': VERSION, 'genome': champion['genome'], 'indexing': INDEX_DOC,
                'CFG': cfg, 'generation': champion['generation'], 'difficulty': cfg['difficulty'],
                'opening_book': JS_OPENING_BOOK, 'evaluation': report or {'complete': False},
                'stage2_accepted': accepted, 'promotion_protocol':state.get('promotion_protocol','legacy-unverified'),
                'champion_verified':state.get('champion_verified',False),
                'champion_validation':state.get('champion_validation')}
    return publish_champion(cfg, artifact, engine_source)

# BEGIN GENERATED CHESS ENGINE BUNDLE
# Generated by tools/sync_engine.py; edit chess_ai/ instead.
def _install_chess_engine_bundle():
    import base64, zlib, json, sys, types
    name = 'chess_ai_bundle_19df73d1965c17dd'
    if name in sys.modules:
        return sys.modules[name]
    sources = json.loads(zlib.decompress(base64.b64decode('...')))
    package = types.ModuleType(name)
    package.__path__ = []
    package.source_sha256 = '19df73d1965c17dda3ab6036bec42f63be21862292e8fbd8bc61efaa6bb5ce03'
    sys.modules[name] = package
    for short in ('board', 'evaluation', 'search', 'neural', 'inference', 'runtime', 'matches', 'training', 'notebook_bridge', 'datasets', 'validation_cases', 'full_pipeline'):
        module = types.ModuleType(name + '.' + short)
        module.__package__ = name
        sys.modules[module.__name__] = module
        setattr(package, short, module)
        exec(compile(sources[short], '<bundled-chess-ai/' + short + '.py>', 'exec'), module.__dict__)
    return package
_CHESS_ENGINE_PACKAGE = _install_chess_engine_bundle()
# END GENERATED CHESS ENGINE BUNDLE
'''
import __future__
exec(compile(CORE_SOURCE, "<core.py>", "exec", flags=__future__.annotations.compiler_flag), globals())

baseline=JSBotPlayer(CFG['seed']); rng=random.Random(CFG['seed']+1); board=chess.Board()
for index in range(200):
    if board.is_game_over(claim_draw=True): board=chess.Board()
    assert baseline.get_move(board) in board.legal_moves
    board.push(rng.choice(list(board.legal_moves)))
print('PASS: JS baseline returned legal moves for 200 randomly reached positions.')

EVAL_SOURCE = r'''
# a1=0..h8=63, always oriented from the owner's side (Black uses square XOR 56).
GENOME_SIZE = 787
INDEX_DOC = {
    '0:5': 'material P,N,B,R,Q; king fixed at zero static material (mate score 30000)',
    '5:389': 'MG PST: index=5+(piece_type-1)*64+owner_square; P,N,B,R,Q,K',
    '389:773': 'EG PST: index=389+(piece_type-1)*64+owner_square; P,N,B,R,Q,K',
    '773:779': 'attack mobility P,N,B,R,Q,K, friendly occupied squares excluded',
    '779': 'bishop pair', '780': 'rook open file', '781': 'rook semi-open file',
    '782': 'doubled pawn penalty per extra pawn', '783': 'isolated pawn penalty per pawn',
    '784': 'passed pawn bonus times (1+relative_rank/3)',
    '785': 'king shield bonus per pawn one rank forward, MG only',
    '786': 'tempo bonus for side to move'}
LOW = np.full(GENOME_SIZE, -180., dtype=np.float64)
HIGH = np.full(GENOME_SIZE, 180., dtype=np.float64)
LOW[:5] = [70,220,230,350,650]; HIGH[:5] = [140,450,460,700,1250]
LOW[773:779] = 0; HIGH[773:779] = [8,20,20,15,10,8]
LOW[779:] = 0; HIGH[779:] = [100,80,50,60,60,100,60,40]
SCALE = np.full(GENOME_SIZE, 25., dtype=np.float64)
SCALE[:5] = [15,35,35,50,90]; SCALE[773:779] = [2,3,3,2,2,2]; SCALE[779:] = 12

def genome_default():
    g = np.zeros(GENOME_SIZE)
    g[:5] = [100,320,335,500,900]
    # Original smooth, handcrafted initialization; not copied PeSTO table data.
    for pt in range(1,7):
        for sq in range(64):
            f,r = chess.square_file(sq), chess.square_rank(sq)
            center = 3.5 - (abs(f-3.5)+abs(r-3.5))/2
            mg = {1:8*r+2*center,2:18*center-35,3:10*center,4:3*r,5:4*center,6:-14*center-5*r}[pt]
            eg = {1:14*r,2:12*center-20,3:10*center,4:4*center,5:6*center,6:18*center}[pt]
            g[5+(pt-1)*64+sq] = mg
            g[389+(pt-1)*64+sq] = eg
    g[773:779] = [1,4,4,2,1,1]
    g[779:] = [30,20,10,15,12,20,12,10]
    return g

def genome_random(sigma=1.0, rng=None):
    rng = rng if rng is not None else np.random.default_rng()
    return np.clip(genome_default()+rng.normal(size=GENOME_SIZE)*SCALE*sigma, LOW, HIGH)

def static_evaluate(board, genome):
    if isinstance(genome, dict) and genome.get('kind') == 'nn':
        return nn_evaluate(board, genome)
    g = genome
    phase = min(24, sum(len(board.pieces(pt,c))*w for pt,w in ((2,1),(3,1),(4,2),(5,4)) for c in chess.COLORS))/24
    score = 0.
    for color in chess.COLORS:
        sign = 1 if color == chess.WHITE else -1
        own = board.occupied_co[color]
        pawns = board.pieces_mask(chess.PAWN,color)
        enemy_pawns = board.pieces_mask(chess.PAWN,not color)
        files = [chess.popcount(pawns & mask) for mask in chess.BB_FILES]
        side = 0.
        for pt in range(1,7):
            squares = board.pieces_mask(pt,color)
            if pt < 6: side += chess.popcount(squares)*g[pt-1]
            for sq in chess.scan_forward(squares):
                oriented = sq if color else sq ^ 56
                side += phase*g[5+(pt-1)*64+oriented]+(1-phase)*g[389+(pt-1)*64+oriented]
                side += g[772+pt]*chess.popcount(board.attacks_mask(sq)&~own)
                if pt == chess.ROOK:
                    file_mask = chess.BB_FILES[chess.square_file(sq)]
                    if not pawns & file_mask:
                        side += g[781] if enemy_pawns & file_mask else g[780]
                if pt == chess.PAWN:
                    f,r = chess.square_file(sq),chess.square_rank(sq)
                    neighbors = (chess.BB_FILES[f-1] if f else 0)|(chess.BB_FILES[f+1] if f<7 else 0)
                    if not pawns & neighbors: side -= g[783]
                    forward = (chess.BB_ALL << (8*(r+1))) & chess.BB_ALL if color else (1 << (8*r))-1
                    if not enemy_pawns & (neighbors|chess.BB_FILES[f]) & forward:
                        relative_rank = r if color else 7-r
                        side += g[784]*(1+relative_rank/3)
        if len(board.pieces(chess.BISHOP,color))>=2: side += g[779]
        side -= g[782]*sum(max(0,n-1) for n in files)
        king = board.king(color)
        if king is not None:
            f,r = chess.square_file(king),chess.square_rank(king)+(1 if color else -1)
            if 0<=r<8:
                side += phase*g[785]*sum(bool(pawns & chess.BB_SQUARES[chess.square(ff,r)]) for ff in range(max(0,f-1),min(8,f+2)))
        score += sign*side
    return float(score if board.turn else -score)+float(g[786])

def evaluate(board, genome):
    if board.is_checkmate(): return -MATE
    if board.is_stalemate() or board.is_insufficient_material() or board.is_fifty_moves() or board.is_repetition(3): return 0.
    return static_evaluate(board, genome)

# Optional stage-2 evaluator: absolute White features, antisymmetrized for exact color symmetry.
def nn_features(board):
    x = np.zeros(768, dtype=np.float32)
    for sq,p in board.piece_map().items():
        x[((0 if p.color else 6)+p.piece_type-1)*64+sq] = 1
    return x

def nn_parameters(vector):
    v = np.asarray(vector, dtype=np.float32); i=0; result=[]
    for shape in ((768,64),(64,),(64,32),(32,),(32,1),(1,)):
        n=math.prod(shape); result.append(v[i:i+n].reshape(shape)); i+=n
    return result

def nn_raw(x, params):
    w1,b1,w2,b2,w3,b3 = params
    return ((np.maximum(0,np.maximum(0,x@w1+b1)@w2+b2)@w3+b3).reshape(-1))

def nn_evaluate(board, genome):
    params = genome.get('_params')
    if params is None:
        params = nn_parameters(genome['weights']); genome['_params'] = params
    x=np.stack((nn_features(board),nn_features(board.mirror())))
    white = float(np.diff(nn_raw(x,params))[0]) * -50
    return float(np.clip(white if board.turn else -white,-MATE+1000,MATE-1000))
'''
import __future__
exec(compile(EVAL_SOURCE, "<evaluation.py>", "exec", flags=__future__.annotations.compiler_flag), globals())


g=genome_default(); positions=[]; rng=random.Random(771); board=chess.Board()
for _ in range(200):
    if board.is_game_over(claim_draw=True): board=chess.Board()
    positions.append(board.copy(stack=False))
    assert abs(evaluate(board,g)-evaluate(board.mirror(),g))<1e-6
    board.push(rng.choice(list(board.legal_moves)))
assert evaluate(chess.Board('7k/6Q1/5K2/8/8/8/8/8 b - - 0 1'),g)==-MATE
assert evaluate(chess.Board('7k/5Q2/6K1/8/8/8/8/8 b - - 0 1'),g)==0
started=time.perf_counter()
for _ in range(5):
    for board in positions: static_evaluate(board,g)
print('Static evaluations/sec:',round(1000/(time.perf_counter()-started)))
print('PASS: symmetry, mate and stalemate.')

SEARCH_SOURCE = r'''
class SearchStopped(Exception): pass

class AlphaBeta:
    def __init__(self, genome, time_limit_s, node_limit):
        self.genome=genome; self.deadline=time.monotonic()+max(.001,time_limit_s)
        self.limit=max(1,int(node_limit)); self.nodes=0; self.tt={}; self.killers={}; self.history=Counter()
        self.counts=Counter()
    def tick(self):
        if self.nodes>=self.limit or time.monotonic()>=self.deadline: raise SearchStopped()
        self.nodes+=1
    def ordered(self, board, moves, ttmove, ply):
        def priority(m):
            if m==ttmove: return 10**9
            victim=board.piece_type_at(m.to_square) or (1 if board.is_en_passant(m) else 0)
            values=(0,100,320,335,500,900,20000)
            if victim or m.promotion:
                return 10**6+16*values[victim]-values[board.piece_type_at(m.from_square)]+values[m.promotion or 0]
            return (500000 if m in self.killers.get(ply,[]) else 0)+self.history[(board.turn,m.from_square,m.to_square)]
        return sorted(moves,key=priority,reverse=True)
    def push(self, board, move):
        board.push(move); key=chess.polyglot.zobrist_hash(board); self.counts[key]+=1
        return key
    def pop(self, board, key):
        self.counts[key]-=1
        if not self.counts[key]: del self.counts[key]
        board.pop()
    def drawn(self,board):
        return board.halfmove_clock>=100 or board.is_insufficient_material() or self.counts[chess.polyglot.zobrist_hash(board)]>=3
    def qsearch(self,board,alpha,beta,ply,qdepth=0):
        self.tick()
        moves=list(board.legal_moves); checked=board.is_check()
        if not moves: return -MATE+ply if checked else 0
        if self.drawn(board): return 0
        # Avoid unbounded checking chains; never stand pat while in check.
        if ply>=96: return 0 if checked else static_evaluate(board,self.genome)
        if not checked:
            stand=static_evaluate(board,self.genome)
            if stand>=beta: return stand
            alpha=max(alpha,stand)
            if qdepth>=8: return alpha
            moves=[m for m in moves if board.is_capture(m) or m.promotion]
        for move in self.ordered(board,moves,None,ply):
            key=self.push(board,move)
            try: value=-self.qsearch(board,-beta,-alpha,ply+1,qdepth+1)
            finally: self.pop(board,key)
            if value>=beta: return value
            alpha=max(alpha,value)
        return alpha
    def negamax(self,board,depth,alpha,beta,ply):
        if depth<=0: return self.qsearch(board,alpha,beta,ply)
        self.tick()
        moves=list(board.legal_moves)
        if not moves: return -MATE+ply if board.is_check() else 0
        if self.drawn(board): return 0
        # Include draw history: equal FENs with different repetition rights are not interchangeable.
        key=(chess.polyglot.zobrist_hash(board),board.halfmove_clock,tuple(sorted(self.counts.items())))
        hit=self.tt.get(key); original_alpha=alpha; ttmove=hit[3] if hit else None
        if hit and hit[0]>=depth:
            value=hit[1]; value=value-ply if value>MATE-1000 else value+ply if value<-MATE+1000 else value
            if hit[2]=='exact' or hit[2]=='lower' and value>=beta or hit[2]=='upper' and value<=alpha: return value
        best=-math.inf; bestmove=moves[0]
        for move in self.ordered(board,moves,ttmove,ply):
            quiet=not board.is_capture(move) and not move.promotion
            childkey=self.push(board,move)
            try: value=-self.negamax(board,depth-1,-beta,-alpha,ply+1)
            finally: self.pop(board,childkey)
            if value>best: best,bestmove=value,move
            alpha=max(alpha,value)
            if alpha>=beta:
                if quiet:
                    self.killers[ply]=([move]+[m for m in self.killers.get(ply,[]) if m!=move])[:2]
                    self.history[(board.turn,move.from_square,move.to_square)]+=depth*depth
                break
        flag='upper' if best<=original_alpha else 'lower' if best>=beta else 'exact'
        stored=best+ply if best>MATE-1000 else best-ply if best<-MATE+1000 else best
        if len(self.tt)>100000: self.tt.clear()
        self.tt[key]=(depth,stored,flag,bestmove)
        return best
    def run(self,board,max_depth):
        board=board.copy(stack=True)
        legal=list(board.legal_moves)
        if not legal: return None,(-MATE if board.is_check() else 0),0,0
        history=board.copy(stack=True)
        while True:
            self.counts[chess.polyglot.zobrist_hash(history)]+=1
            if not history.move_stack: break
            history.pop()
        best,score,completed=legal[0],static_evaluate(board,self.genome),0
        if self.drawn(board): return best,0,0,0
        for depth in range(1,max_depth+1):
            iteration_best,iteration_score=best,-math.inf
            try:
                alpha=-MATE-1
                for move in self.ordered(board,legal,best,0):
                    key=self.push(board,move)
                    try: value=-self.negamax(board,depth-1,-MATE-1,-alpha,1)
                    finally: self.pop(board,key)
                    if value>iteration_score: iteration_best,iteration_score=move,value
                    alpha=max(alpha,value)
                best,score,completed=iteration_best,iteration_score,depth
                if abs(score)>MATE-1000: break
            except SearchStopped: break
        return best,float(score),completed,self.nodes

def search(board, genome, max_depth=3, time_limit_s=.5, node_limit=100000):
    return AlphaBeta(genome,time_limit_s,node_limit).run(board,max_depth)

DEFAULT_DIFFICULTY = {
    'easy': {'depth':2,'seconds':.15,'nodes':2000,'temperature':120.,'inaccuracy':.12},
    'medium': {'depth':3,'seconds':.5,'nodes':30000,'temperature':0.,'inaccuracy':0.},
    'hard': {'depth':64,'seconds':2.5,'nodes':180000,'temperature':0.,'inaccuracy':0.}}

class CristopherEngine:
    def __init__(self, model_path, seed=42):
        data=json.loads(Path(model_path).read_text())
        if data['version']!=VERSION: raise ValueError('Incompatible model version')
        self.genome=np.asarray(data['genome'],dtype=float)
        self.nn=data.get('stage2_accepted')
        self.settings=data.get('difficulty',DEFAULT_DIFFICULTY)
        self.book=data.get('opening_book',JS_OPENING_BOOK)
        self.rng=np.random.default_rng(seed)
    def get_move(self, fen: str, history: list[str] | None = None, difficulty: str = 'medium') -> dict:
        started=time.monotonic(); board=board_from_history(fen,history)
        if difficulty not in self.settings: raise ValueError('Choose easy, medium, or hard')
        legal=list(board.legal_moves)
        if not legal: raise ValueError('Game over: no legal move')
        genome=self.nn or self.genome
        move=None; score=0.; depth=0; nodes=0
        for candidate in legal:
            board.push(candidate); mate=board.is_checkmate(); board.pop()
            if mate: move=candidate; score=MATE-1; depth=1; break
        book=[chess.Move.from_uci(u) for u in self.book.get(fen_prefix(board),[]) if u in {m.uci() for m in legal}]
        if move is None and book and len(board.move_stack)<12 and board.fullmove_number<=6:
            move=book[int(self.rng.integers(len(book)))]
            board.push(move); score=-evaluate(board,genome); board.pop()
        if move is None:
            params=self.settings[difficulty]
            move,score,depth,nodes=search(board,genome,params['depth'],params['seconds'],params['nodes'])
            # Easy uses static top-move sampling, while preserving forced mate priority above.
            if params['temperature']>0:
                ranked=[]
                for candidate in legal:
                    board.push(candidate); value=-evaluate(board,genome); board.pop()
                    ranked.append((candidate,value))
                ranked.sort(key=lambda item:item[1],reverse=True)
                pool=ranked if self.rng.random()<params['inaccuracy'] else ranked[:min(5,len(ranked))]
                logits=np.array([v for _,v in pool])/params['temperature']; weights=np.exp(np.clip(logits-logits.max(),-60,0)); weights/=weights.sum()
                move,score=pool[int(self.rng.choice(len(pool),p=weights))]; depth=0
        if history and (score>50 or static_evaluate(board,genome)>50):
            board.push(move); repeats=board.is_repetition(3); board.pop()
            if repeats:
                safe=[]
                for candidate in legal:
                    board.push(candidate)
                    repeated=board.is_repetition(3)
                    value=-evaluate(board,genome)
                    board.pop()
                    if not repeated: safe.append((value,candidate))
                if safe: score,move=max(safe,key=lambda pair:pair[0]); depth=0
        assert move in board.legal_moves
        return {'move':move.uci(),'info':{'eval':float(score),'depth':int(depth),'nodes':int(nodes),'time_ms':int((time.monotonic()-started)*1000)}}
'''
import __future__
exec(compile(SEARCH_SOURCE, "<search.py>", "exec", flags=__future__.annotations.compiler_flag), globals())

if CFG['training_mode']=='classic':
    g=genome_default(); started=time.perf_counter(); total_nodes=0
    for board in positions[:12]:
        move,score,depth,nodes=search(board,g,2,.2,20000)
        assert move in board.legal_moves
        total_nodes+=nodes
    elapsed=time.perf_counter()-started
    print('Nodes/sec:',round(total_nodes/elapsed),'| estimated 80-ply depth-2 games/min/core:',round(60/(elapsed/12*80),2))
    print('If this is below 20, consider the optional compiled backend below. The Python fallback remains supported.')
else:
    print('Skipping the classic CPU search benchmark; GPU training uses batched successor evaluation.')


CPP_SOURCE = r'''
#include "chess.hpp"
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstring>
#include <map>
#include <sstream>
#include <unordered_map>
#include <vector>
using namespace chess;
using Clock=std::chrono::steady_clock;
constexpr double MATE=30000;
Bitboard attack(const Board& b,PieceType pt,Square sq,Color c){
 if(pt==PieceType::PAWN)return attacks::pawn(c,sq);
 if(pt==PieceType::KNIGHT)return attacks::knight(sq);
 if(pt==PieceType::BISHOP)return attacks::bishop(sq,b.occ());
 if(pt==PieceType::ROOK)return attacks::rook(sq,b.occ());
 if(pt==PieceType::QUEEN)return attacks::bishop(sq,b.occ())|attacks::rook(sq,b.occ());
 return attacks::king(sq);
}
double eval(const Board& b,const double*g){
 double phase=0,total=0;int pw[]={0,1,1,2,4,0};
 for(int t=0;t<6;t++)phase+=b.pieces(PieceType(static_cast<PieceType::underlying>(t))).count()*pw[t];
 phase=std::min(24.,phase)/24.;
 for(Color c:{Color::WHITE,Color::BLACK}){
  double side=0;int files[8]={};auto pawns=b.pieces(PieceType::PAWN,c);auto ep=b.pieces(PieceType::PAWN,~c);
  for(int s=0;s<64;s++)if(pawns.check(s))files[s%8]++;
  for(int t=0;t<6;t++){
   PieceType pt(static_cast<PieceType::underlying>(t));auto pieces=b.pieces(pt,c);
   if(t<5)side+=pieces.count()*g[t];
   while(pieces){int s=pieces.pop();Square sq(s);int f=s%8,r=s/8,ori=c==Color::WHITE?s:s^56;
    side+=phase*g[5+t*64+ori]+(1-phase)*g[389+t*64+ori];
    side+=g[773+t]*(attack(b,pt,sq,c)&~b.us(c)).count();
    if(t==3 && !files[f]){bool enemy=false;for(int rr=0;rr<8;rr++)enemy|=ep.check(rr*8+f);side+=g[enemy?781:780];}
    if(t==0){
     if((f==0 || !files[f-1])&&(f==7 || !files[f+1]))side-=g[783];
     bool passed=true;for(int rr=0;rr<8;rr++)if(c==Color::WHITE?rr>r:rr<r)
      for(int ff=std::max(0,f-1);ff<=std::min(7,f+1);ff++)if(ep.check(rr*8+ff))passed=false;
     if(passed)side+=g[784]*(1+(c==Color::WHITE?r:7-r)/3.);
    }
   }
  }
  if(b.pieces(PieceType::BISHOP,c).count()>=2)side+=g[779];
  for(int f=0;f<8;f++)side-=g[782]*std::max(0,files[f]-1);
  int k=b.kingSq(c).index(),r=k/8+(c==Color::WHITE?1:-1),f=k%8;
  if(r>=0&&r<8)for(int ff=std::max(0,f-1);ff<=std::min(7,f+1);ff++)if(pawns.check(r*8+ff))side+=phase*g[785];
  total+=(c==Color::WHITE?1:-1)*side;
 }
 return (b.sideToMove()==Color::WHITE?total:-total)+g[786];
}
struct Stop{};
struct Entry{int depth;double value;int flag;Move move;};
struct Search{
 const double*g;Clock::time_point deadline;int limit,nodes=0;std::map<std::uint64_t,int> counts;
 std::unordered_map<std::uint64_t,Entry>tt;Move killers[100][2]{};int history[2][64][64]{};
 Search(const double*w,double seconds,int maxnodes):g(w),deadline(Clock::now()+std::chrono::microseconds((long long)(seconds*1e6))),limit(maxnodes){}
 void tick(){if(nodes>=limit||Clock::now()>=deadline)throw Stop{};nodes++;}
 bool draw(const Board&b){return b.halfMoveClock()>=100||b.isInsufficientMaterial()||counts[b.hash()]>=3;}
 std::uint64_t key(const Board&b){auto h=b.hash()^(std::uint64_t(b.halfMoveClock())<<48);for(auto [k,n]:counts)h^=(k*0x9e3779b97f4a7c15ULL)^(std::uint64_t(n)*((k>>1)|1));return h;}
 void push(Board&b,Move m){b.makeMove(m);counts[b.hash()]++;}
 void pop(Board&b,Move m){auto h=b.hash();if(--counts[h]==0)counts.erase(h);b.unmakeMove(m);}
 std::vector<Move> ordered(const Board&b,const Movelist&list,Move preferred,int ply){
  std::vector<Move> out(list.begin(),list.end());int val[]={100,320,335,500,900,20000,0};
  auto rank=[&](Move m){if(m==preferred)return 1000000000;int victim=b.at(m.to()).type();
   if(m.typeOf()==Move::ENPASSANT)victim=0;
   if(b.isCapture(m)||m.typeOf()==Move::PROMOTION)return 1000000+16*val[victim]-val[b.at(m.from()).type()]+(m.typeOf()==Move::PROMOTION?val[m.promotionType()]:0);
   return ((m==killers[ply][0]||m==killers[ply][1])?500000:0)+history[b.sideToMove()][m.from().index()][m.to().index()];};
  std::stable_sort(out.begin(),out.end(),[&](Move a,Move z){return rank(a)>rank(z);});return out;
 }
 double q(Board&b,double a,double beta,int ply,int qdepth){
  tick();Movelist list;movegen::legalmoves(list,b);bool check=b.inCheck();
  if(list.empty())return check?-MATE+ply:0;if(draw(b))return 0;
  if(ply>=96)return check?0:eval(b,g);
  if(!check){double v=eval(b,g);if(v>=beta)return v;a=std::max(a,v);if(qdepth>=8)return a;}
  for(auto m:ordered(b,list,Move(),ply)){
   if(!check&&!b.isCapture(m)&&m.typeOf()!=Move::PROMOTION)continue;
   push(b,m);double v;try{v=-q(b,-beta,-a,ply+1,qdepth+1);}catch(...){pop(b,m);throw;}pop(b,m);
   if(v>=beta)return v;a=std::max(a,v);
  }return a;
 }
 double ab(Board&b,int depth,double a,double beta,int ply){
  if(depth<=0)return q(b,a,beta,ply,0);tick();Movelist list;movegen::legalmoves(list,b);
  if(list.empty())return b.inCheck()?-MATE+ply:0;if(draw(b))return 0;
  auto k=key(b);auto it=tt.find(k);Move preferred;double olda=a;
  if(it!=tt.end()){
   auto e=it->second;preferred=e.move;double v=e.value;v=v>MATE-1000?v-ply:v<-MATE+1000?v+ply:v;
   if(e.depth>=depth && (e.flag==0||(e.flag==1&&v>=beta)||(e.flag==2&&v<=a)))return v;
  }
  double best=-MATE-1;Move bestmove=list[0];
  for(auto m:ordered(b,list,preferred,ply)){
   bool quiet=!b.isCapture(m)&&m.typeOf()!=Move::PROMOTION;push(b,m);double v;
   try{v=-ab(b,depth-1,-beta,-a,ply+1);}catch(...){pop(b,m);throw;}pop(b,m);
   if(v>best){best=v;bestmove=m;}a=std::max(a,v);
   if(a>=beta){if(quiet){killers[ply][1]=killers[ply][0];killers[ply][0]=m;history[b.sideToMove()][m.from().index()][m.to().index()]+=depth*depth;}break;}
  }
  if(tt.size()>100000)tt.clear();double stored=best>MATE-1000?best+ply:best<-MATE+1000?best-ply:best;
  tt[k]={depth,stored,best<=olda?2:best>=beta?1:0,bestmove};return best;
 }
};
extern "C" double evo_evaluate(const char*fen,const double*g){try{Board b(fen);return eval(b,g);}catch(...){return NAN;}}
extern "C" int evo_search(const char*fen,const char*moves,const double*g,int depth,double seconds,int limit,char*out,double*score,int*completed,int*nodes){
 try{
  Board b(fen);Search s(g,seconds,limit);s.counts[b.hash()]++;std::istringstream stream(moves);std::string u;
  while(stream>>u){auto m=uci::uciToMove(b,u);b.makeMove(m);s.counts[b.hash()]++;}
  Movelist list;movegen::legalmoves(list,b);if(list.empty())return 1;
  Move best=list[0];*score=eval(b,g);*completed=0;
  if(s.draw(b))*score=0;
  else for(int d=1;d<=depth;d++){
   Move candidate=best;double value=-MATE-1,a=-MATE-1;
   try{
    for(auto m:s.ordered(b,list,best,0)){
     s.push(b,m);double v;try{v=-s.ab(b,d-1,-MATE-1,-a,1);}catch(...){s.pop(b,m);throw;}s.pop(b,m);
     if(v>value){value=v;candidate=m;}a=std::max(a,v);
    }
    best=candidate;*score=value;*completed=d;if(std::abs(value)>MATE-1000)break;
   }catch(Stop&){break;}
  }
  *nodes=s.nodes;std::strcpy(out,uci::moveToUci(best).c_str());return 0;
 }catch(...){return 2;}
}
'''

CPP_WRAPPER_SOURCE = r'''
# Optional acceleration. The standalone export works without this library.
import ctypes
_CPP_HANDLES = {}
def cpp_library(path):
    if path not in _CPP_HANDLES:
        lib=ctypes.CDLL(path); ptr=ctypes.POINTER(ctypes.c_double)
        lib.evo_evaluate.argtypes=[ctypes.c_char_p,ptr]; lib.evo_evaluate.restype=ctypes.c_double
        lib.evo_search.argtypes=[ctypes.c_char_p,ctypes.c_char_p,ptr,ctypes.c_int,ctypes.c_double,ctypes.c_int,
                                ctypes.c_char_p,ptr,ctypes.POINTER(ctypes.c_int),ctypes.POINTER(ctypes.c_int)]
        lib.evo_search.restype=ctypes.c_int; _CPP_HANDLES[path]=lib
    return _CPP_HANDLES[path]

def cpp_search(board,genome,max_depth,time_limit_s,node_limit,path):
    g=np.ascontiguousarray(genome,dtype=np.float64)
    if g.shape!=(GENOME_SIZE,): raise ValueError('Wrong genome size')
    root=board.root(); moves=' '.join(m.uci() for m in board.move_stack)
    out=ctypes.create_string_buffer(8); score=ctypes.c_double(); depth=ctypes.c_int(); nodes=ctypes.c_int()
    status=cpp_library(path).evo_search(root.fen().encode(),moves.encode(),g.ctypes.data_as(ctypes.POINTER(ctypes.c_double)),
        int(max_depth),float(time_limit_s),int(node_limit),out,ctypes.byref(score),ctypes.byref(depth),ctypes.byref(nodes))
    if status==1: return None,(-MATE if board.is_checkmate() else 0),0,0
    if status: raise RuntimeError('C++ search failed; disable cpp_backend and use a new run_name.')
    move=chess.Move.from_uci(out.value.decode())
    if move not in board.legal_moves: raise RuntimeError('C++ backend returned an illegal move')
    return move,score.value,depth.value,nodes.value

python_search=search
def search(board,genome,max_depth=3,time_limit_s=.5,node_limit=100000):
    path=os.environ.get('CRISTOPHER_CPP_PATH','')
    if path and Path(path).is_file() and not isinstance(genome,dict):
        return cpp_search(board,genome,max_depth,time_limit_s,node_limit,path)
    return python_search(board,genome,max_depth,time_limit_s,node_limit)
'''
import __future__
exec(compile(CPP_WRAPPER_SOURCE, "<cpp_wrapper.py>", "exec", flags=__future__.annotations.compiler_flag), globals())

# Preserve the backend of a saved run; changing its search body would mix experiments.
for checkpoint_path in sorted((ROOT/'checkpoints'/CFG['run_name']).glob('generation_*.json'), reverse=True):
    try:
        saved_cfg = json.loads(checkpoint_path.read_text())['CFG']
    except (OSError, ValueError, KeyError):
        continue
    saved_backend = saved_cfg.get('cpp_backend', False)
    if CFG['cpp_backend'] != saved_backend:
        print('Keeping the saved run backend:', 'C++' if saved_backend else 'Python',
              '| Choose a new run_name to change backends.')
        CFG['cpp_backend'] = saved_backend
    break

if CFG['cpp_backend']:
    import urllib.request
    try:
        commit='2a608d58f44776909bb6e2c09505e97ba1451319'
        for remote,local in (('include/chess.hpp','chess.hpp'),('LICENSE','chess-library-LICENSE')):
            target=MODELS/local
            if not target.exists():
                urllib.request.urlretrieve(f'https://raw.githubusercontent.com/Disservin/chess-library/{commit}/{remote}',target)
        assert hashlib.sha256((MODELS/'chess.hpp').read_bytes()).hexdigest()=='905b95c2ac870e488a5262c992e3aed0907a2233eb95b9a2835a3d47d9a3cf5d'
        get_ipython().run_cell_magic('writefile',str(MODELS/'cristopher_backend.cpp'),CPP_SOURCE)
        binary=MODELS/'cristopher_backend.so'
        subprocess.check_call(['g++','-std=c++17','-O3','-DNDEBUG','-shared','-fPIC',str(MODELS/'cristopher_backend.cpp'),'-o',str(binary)])
        for board in positions[:40]:
            vector=np.ascontiguousarray(genome_default())
            actual=cpp_library(str(binary)).evo_evaluate(board.fen().encode(),vector.ctypes.data_as(ctypes.POINTER(ctypes.c_double)))
            assert abs(actual-static_evaluate(board,vector))<1e-6
            answer=cpp_search(board,vector,2,.1,10000,str(binary))
            assert answer[0] in board.legal_moves
        os.environ['CRISTOPHER_CPP_PATH']=str(binary)
        print('Compiled backend enabled; evaluation parity and legal search checks passed.')
    except Exception as error:
        os.environ.pop('CRISTOPHER_CPP_PATH',None)
        CFG['cpp_backend']=False
        print('Compiled backend unavailable; using Python:',error)
else:
    os.environ.pop('CRISTOPHER_CPP_PATH',None)
    print('Python search enabled.')

GA_SOURCE = r'''
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED
import multiprocessing as mp
import importlib

def individual(genome, sigma=.25):
    return {'genome':np.asarray(genome,dtype=float).tolist(),'sigma':float(sigma)}

def initial_population(cfg,rng):
    return [individual(genome_default(),cfg['mutation_sigma'])]+[
        individual(genome_random(1.0 if i>=int(cfg['pop_size']*.75) else cfg['mutation_sigma'],rng),cfg['mutation_sigma'])
        for i in range(1,cfg['pop_size'])]

def diversity(pop):
    return float(np.mean(np.std(np.asarray([p['genome'] for p in pop])/SCALE,axis=0)))

def next_population(pop,fitness,cfg,rng):
    values=np.asarray([p['genome'] for p in pop])/SCALE
    distances=np.sqrt(np.mean((values[:,None]-values[None,:])**2,axis=2)); np.fill_diagonal(distances,np.inf)
    novelty=distances.min(axis=1)
    converged=diversity(pop)<cfg['diversity_floor']
    selection=np.asarray(fitness)+(cfg['novelty_weight']*novelty if converged else 0)
    elite=np.argsort(fitness)[-cfg['elite']:][::-1]
    result=[json.loads(json.dumps(pop[int(i)])) for i in elite]
    def parent():
        choices=rng.choice(len(pop),min(cfg['tournament'],len(pop)),replace=False)
        return pop[int(max(choices,key=lambda i:selection[i]))]
    while len(result)<len(pop):
        a,b=parent(),parent(); x,y=np.array(a['genome']),np.array(b['genome'])
        if rng.random()<.5:
            alpha=cfg['blx_alpha']; lo=np.minimum(x,y); span=np.abs(x-y)
            child=rng.uniform(lo-alpha*span,lo+(1+alpha)*span)
        else: child=np.where(rng.random(GENOME_SIZE)<.5,x,y)
        sigma=float(np.clip(math.sqrt(a['sigma']*b['sigma'])*math.exp(rng.normal(0,.15)),.02,1.5))
        if converged: sigma=min(1.5,sigma*1.3)
        mask=rng.random(GENOME_SIZE)<cfg['mutation_rate']
        child+=mask*rng.normal(size=GENOME_SIZE)*SCALE*sigma
        result.append(individual(np.clip(child,LOW,HIGH),sigma))
    return result

def player(kind,genome=None,**kwargs):
    return {'kind':kind,**({'genome':genome} if genome is not None else {}),**kwargs}

def game_task(task):
    cfg=task['cfg']; board=chess.Board()
    for uci in task['opening']: board.push_uci(uci)
    rng=random.Random(task['seed']); js=JSBotPlayer(task['seed'])
    engines={}; brains={}; reason=''; winner=None; truncated=False
    contestants=[task['white'],task['black']]
    try:
        for side,spec in enumerate(contestants):
            if spec['kind']=='stockfish':
                sf=chess.engine.SimpleEngine.popen_uci(cfg['stockfish'],timeout=5)
                engines[side]=sf
                option=sf.options['UCI_Elo']; level=max(option.min,min(option.max,int(spec['elo'])))
                sf.configure({'Threads':1,'Hash':16,'UCI_LimitStrength':True,'UCI_Elo':level})
        while not board.is_game_over(claim_draw=True):
            if time.time()>=task['deadline']: return None
            if board.ply()>=cfg['max_plies']:
                truncated=True; reason='ply limit: unfinished, neutral fitness; excluded from Elo'; break
            side=0 if board.turn else 1; spec=contestants[side]; began=time.monotonic()
            if spec['kind']=='js': move=js.get_move(board)
            elif spec['kind']=='random': move=rng.choice(list(board.legal_moves))
            elif spec['kind']=='stockfish':
                try: move=engines[side].play(board,chess.engine.Limit(time=cfg['stockfish_seconds'])).move
                except (chess.engine.EngineError,TimeoutError):
                    winner=not board.turn; reason='engine failure'; break
            else:
                genome=spec['genome'] if isinstance(spec['genome'],dict) else np.asarray(spec['genome'])
                if 'difficulty_name' in cfg:
                    if side not in brains:
                        brain=CristopherEngine.__new__(CristopherEngine)
                        brain.genome=genome; brain.nn=genome if isinstance(genome,dict) else None
                        brain.settings=cfg['difficulty_settings']; brain.book=JS_OPENING_BOOK
                        brain.rng=np.random.default_rng(task['seed']+side); brains[side]=brain
                    answer=brains[side].get_move(board.fen(),[m.uci() for m in board.move_stack],cfg['difficulty_name'])
                    move=chess.Move.from_uci(answer['move'])
                else:
                    move,_,_,_=search(board,genome,cfg['depth'],cfg['move_seconds'],cfg['node_limit'])
            if time.monotonic()-began>cfg['forfeit_seconds']:
                winner=not board.turn; reason='timeout'; break
            if move not in board.legal_moves:
                winner=not board.turn; reason='illegal move'; break
            board.push(move)
        else:
            outcome=board.outcome(claim_draw=True); winner=outcome.winner; reason=outcome.termination.name
        result='*' if truncated else '1/2-1/2' if winner is None else '1-0' if winner else '0-1'
        score=.5 if winner is None else float(winner==task['candidate_white'])
        bonus=cfg['fast_win_bonus']*max(0,1-board.ply()/cfg['max_plies']) if score==1 else 0
        penalty=cfg['fault_penalty'] if score==0 and reason in ('timeout','illegal move','engine failure') else 0
        row={'id':task['id'],'candidate':task['candidate'],'opponent':task['opponent'],'pair':task['pair'],
             'candidate_white':task['candidate_white'],'score':score,'fitness':score+bonus-penalty,
             'result':result,'plies':board.ply(),'reason':reason,'truncated':truncated,'elo':task.get('elo')}
        game=chess.pgn.Game.from_board(board)
        game.headers.update({'Event':task['event'],'White':contestants[0]['kind'],'Black':contestants[1]['kind'],
                             'Result':result,'Termination':reason,'Seed':str(task['seed'])})
        path=Path(task['pgn']); path.parent.mkdir(parents=True,exist_ok=True)
        temp=path.with_suffix('.tmp'); temp.write_text(str(game)+'\n\n'); os.replace(temp,path)
        atomic_json(task['result_path'],row)
        return row
    finally:
        for sf in engines.values():
            try: sf.quit()
            except Exception: pass

def match_config(cfg, difficulty=None):
    result={key:cfg[key] for key in ('max_plies','depth','move_seconds','node_limit','stockfish','stockfish_seconds','forfeit_seconds','fast_win_bonus','fault_penalty')}
    if difficulty and not cfg['QUICK_TEST']:
        p=cfg['difficulty'][difficulty]; result.update(depth=p['depth'],move_seconds=p['seconds'],node_limit=p['nodes'],difficulty_name=difficulty,difficulty_settings=cfg['difficulty'])
    return result

def paired_tasks(cfg,namespace,candidate_index,candidate,opponent_name,opponent,openings,match_cfg=None):
    result=[]
    for pair,opening in enumerate(openings):
        for white in (True,False):
            label=f'{candidate_index}_{opponent_name}_{pair}_{int(white)}'
            root=Path(cfg['root']); directory=root/'logs'/cfg['run_name']/namespace
            task={'id':label,'candidate':candidate_index,'opponent':opponent_name,'pair':pair,'opening':opening,
                  'seed':cfg['seed']+pair*101,'candidate_white':white,'white':candidate if white else opponent,
                  'black':opponent if white else candidate,'cfg':match_cfg or match_config(cfg),
                  'event':f'Cristopher evolution {namespace}','elo':opponent.get('elo'),
                  'pgn':str(root/'pgn'/cfg['run_name']/namespace/(label+'.pgn')),
                  'result_path':str(directory/(label+'.json')),'deadline':0}
            result.append(task)
    return result

def generation_tasks(cfg,state):
    pop=state['population']; gen=state['generation']
    openings=[random_opening(cfg['seed']+gen*1009+p) for p in range(cfg['pairs_per_opponent'])]
    jobs=[]
    for i,ind in enumerate(pop):
        candidate=player('genome',ind['genome'])
        opponents=[('js',player('js')),('random',player('random'))]
        for offset in range(1,cfg['peer_opponents']+1):
            j=(i+1+(gen+offset-1)%(len(pop)-1))%len(pop)
            opponents.append((f'peer{offset}',player('genome',pop[j]['genome'])))
        hall=state['hall'] or [{'genome':genome_default().tolist()}]
        opponents.append(('hall',player('genome',hall[gen%len(hall)]['genome'])))
        if gen%cfg['anchor_every']==0:
            opponents.append(('stockfish',player('stockfish',elo=cfg['anchor_elo'])))
        for name,opponent in opponents:
            jobs.extend(paired_tasks(cfg,f'g{gen:05d}',i,candidate,name,opponent,openings))
    return jobs

def run_jobs(jobs,cfg,deadline):
    from tqdm.auto import tqdm
    rows=[]; pending=[]
    for job in jobs:
        path=Path(job['result_path'])
        if path.exists(): rows.append(json.loads(path.read_text()))
        else:
            job=dict(job); job['deadline']=deadline; pending.append(job)
    progress=tqdm(total=len(jobs),initial=len(rows),desc='Completed games',mininterval=2)
    if cfg['workers']==1:
        try:
            for job in pending:
                if time.time()>=deadline: break
                row=game_task(job)
                if row is not None: rows.append(row); progress.update()
        except KeyboardInterrupt: print('Interrupted; completed game files will be reused.')
        finally: progress.close()
        return rows
    # Worker functions live in an importable Drive module, never in notebook __main__.
    worker=importlib.import_module('cristopher_evo_worker')
    executor=ProcessPoolExecutor(max_workers=cfg['workers'],mp_context=mp.get_context('spawn'))
    futures={}; cursor=0
    try:
        while cursor<len(pending) or futures:
            while cursor<len(pending) and len(futures)<cfg['workers'] and time.time()<deadline:
                job=pending[cursor]; cursor+=1
                futures[executor.submit(worker.game_task,job)]=job['id']
            if not futures: break
            done,_=wait(futures,timeout=1,return_when=FIRST_COMPLETED)
            for future in done:
                futures.pop(future); row=future.result()
                if row is not None: rows.append(row); progress.update()
            if time.time()>=deadline and not futures: break
    except KeyboardInterrupt:
        print('Stopping workers; completed games stay saved. No unfinished game is scored.')
        for future in futures: future.cancel()
        # A user interrupt must not wait for a full game. Children may leave only .tmp files.
        for process in list(executor._processes.values()): process.terminate()
    finally:
        executor.shutdown(wait=True,cancel_futures=True); progress.close()
    return rows

def signature(cfg):
    excluded={'training_mode','max_hours','workers','run_evaluation','eval_max_hours','STAGE2','quick_play','root','stockfish'}
    return digest({'version':VERSION,'cfg':{k:v for k,v in cfg.items() if k not in excluded and not k.startswith('gpu_')}})

def save_state(cfg,state):
    folder=Path(cfg['root'])/'checkpoints'/cfg['run_name']; folder.mkdir(parents=True,exist_ok=True)
    pack={'version':VERSION,'signature':signature(cfg),'CFG':cfg,'state':state}
    path=folder/f"generation_{state['generation']:05d}.json"
    atomic_json(path,pack)
    for old in sorted(folder.glob('generation_*.json'))[:-3]: old.unlink()
    if state.get('champion'):
        atomic_json(folder/'best_champion.json',state['champion'])
        publish_classic_champion(cfg,state,Path(__file__).read_text())

def load_state(cfg):
    folder=Path(cfg['root'])/'checkpoints'/cfg['run_name']
    paths=sorted(folder.glob('generation_*.json'),reverse=True)
    for path in paths:
        try: pack=json.loads(path.read_text())
        except (OSError,json.JSONDecodeError):
            print('Skipping unreadable checkpoint',path.name); continue
        if pack['signature']!=signature(cfg): raise ValueError('Configuration changed: restore settings or choose a new run_name.')
        print('Resuming generation',pack['state']['generation'],'from',path.name)
        return pack['state']
    if paths: raise RuntimeError('No readable checkpoint; restore a Drive backup.')
    rng=np.random.default_rng(cfg['seed'])
    state={'generation':0,'population':initial_population(cfg,rng),'fitness':None,'hall':[],'champion':None,
           'rng':rng.bit_generator.state,'python_rng':random.Random(cfg['seed']).getstate(),'history':[]}
    save_state(cfg,state); return state

def evolve(cfg):
    state=load_state(cfg); rng=np.random.default_rng(); rng.bit_generator.state=state['rng']
    if not state.get('champion'):
        state['champion']={'genome':genome_default().tolist(),'sigma':cfg['mutation_sigma'],'generation':-1}
        save_state(cfg,state)
    def tuples(x): return tuple(tuples(v) for v in x) if isinstance(x,list) else x
    random.setstate(tuples(state['python_rng']))
    deadline=time.time()+cfg['max_hours']*3600
    print(f"Evolution resources | {cfg['workers']} CPU workers | backend: {'C++' if cfg['cpp_backend'] else 'Python'} | session: {cfg['max_hours']} hours", flush=True)
    while state['generation']<cfg['generations'] and time.time()<deadline:
        gen=state['generation']; print(f"Generation {gen+1}/{cfg['generations']} | population {len(state['population'])}",flush=True)
        jobs=generation_tasks(cfg,state); rows=run_jobs(jobs,cfg,deadline)
        if len(rows)!=len(jobs):
            save_state(cfg,state); print('Partial generation saved by game. Rerun evolution to resume.'); return state
        fitness=[float(np.mean([r['fitness'] for r in rows if r['candidate']==i])) for i in range(len(state['population']))]
        idx=int(np.argmax(fitness)); champion=state['population'][idx]
        promoted=_CHESS_ENGINE_PACKAGE.notebook_bridge.validate_classic(__import__('sys').modules[__name__],cfg,state,champion['genome'],deadline)
        if promoted is None:
            save_state(cfg,state); return state
        def score(name):
            values=[r['score'] for r in rows if r['candidate']==idx and r['opponent']==name]
            return float(np.mean(values)) if values else None
        record={'generation':gen,'best':max(fitness),'mean':float(np.mean(fitness)),
                'diversity':diversity(state['population']),'js':score('js'),'random':score('random'),'stockfish':score('stockfish'),
                'unfinished_fraction':sum(r['truncated'] for r in rows)/len(rows),'promoted':promoted}
        state['history'].append(record); print(record,flush=True)
        # Parent selection uses fitness; deployment promotion uses independent matches.
        if promoted:
            state['hall'].append(dict(state['champion'])); state['hall']=state['hall'][-cfg['hall_size']:]
            state['champion']={'genome':champion['genome'],'sigma':champion['sigma'],'generation':gen,'fitness':fitness[idx]}
        state['evaluated_population']=state['population']; state['fitness']=fitness
        state['population']=next_population(state['population'],fitness,cfg,rng)
        state['generation']+=1; state['rng']=rng.bit_generator.state; state['python_rng']=random.getstate()
        save_state(cfg,state)
        atomic_json(Path(cfg['root'])/'logs'/cfg['run_name']/'generations.json',state['history'])
    print('Evolution session finished. Completed generations:',state['generation'])
    return state
'''
import __future__
exec(compile(GA_SOURCE, "<ga.py>", "exec", flags=__future__.annotations.compiler_flag), globals())

REPORT_SOURCE = r'''
def elo_mle(rows):
    rows=[r for r in rows if not r['truncated'] and r['elo'] is not None]
    if not rows: return None
    ratings=np.array([r['elo'] for r in rows]); scores=np.array([r['score'] for r in rows])
    if scores.sum()==0 or scores.sum()==len(scores): return None
    lo,hi=float(ratings.min()-2000),float(ratings.max()+2000)
    for _ in range(60):
        mid=(lo+hi)/2; expected=1/(1+10**np.clip((ratings-mid)/400,-20,20))
        if expected.sum()<scores.sum(): lo=mid
        else: hi=mid
    return (lo+hi)/2

def elo_summary(rows,seed):
    eligible=[r for r in rows if not r['truncated'] and r['elo'] is not None]
    estimate=elo_mle(eligible)
    if estimate is None: return {'elo':None,'interval':None,'rated_games':len(eligible)}
    pairs={}
    for row in eligible: pairs.setdefault((row['opponent'],row['pair']),[]).append(row)
    groups=list(pairs.values()); rng=np.random.default_rng(seed); estimates=[]
    for _ in range(400):
        sample=[r for i in rng.integers(len(groups),size=len(groups)) for r in groups[int(i)]]
        value=elo_mle(sample)
        if value is not None: estimates.append(value)
    interval=np.percentile(estimates,[2.5,97.5]).tolist() if len(estimates)>=360 else None
    return {'elo':estimate,'interval':interval,'rated_games':len(eligible)}

def comparison(rows):
    finished=[r for r in rows if not r['truncated']]
    groups={}
    for row in finished: groups.setdefault(row['pair'],[]).append(row['score'])
    pairs=[np.mean(v) for v in groups.values() if len(v)==2]
    interval=None
    if len(pairs)>=10:
        rng=np.random.default_rng(981); means=[float(np.mean(rng.choice(pairs,len(pairs)))) for _ in range(1000)]
        interval=np.percentile(means,[2.5,97.5]).tolist()
    return {'wins':sum(r['score']==1 for r in finished),'draws':sum(r['score']==.5 for r in finished),
            'losses':sum(r['score']==0 for r in finished),'unfinished':len(rows)-len(finished),
            'score_interval':interval,'improvement_demonstrated':bool(interval and interval[0]>.5)}

def evaluate_champion(cfg,state):
    if not state.get('champion'): raise RuntimeError('Complete at least one generation first.')
    genome=state['champion']['genome']; candidate=player('genome',genome)
    namespace='evaluation_'+digest({'genome':genome,'signature':signature(cfg)})[:16]
    openings=[random_opening(cfg['seed']+500000+p) for p in range(cfg['eval_games']//2)]
    with chess.engine.SimpleEngine.popen_uci(cfg['stockfish']) as sf:
        option=sf.options['UCI_Elo']; levels=sorted(set(max(option.min,min(option.max,int(v))) for v in cfg['eval_levels']))
    jobs=[]
    for difficulty in cfg['eval_difficulties']:
        mc=match_config(cfg,difficulty)
        opponents=[('js',player('js')),('random',player('random')),('default',player('genome',genome_default().tolist()))]
        opponents += [(f'stockfish_{elo}',player('stockfish',elo=elo)) for elo in levels]
        for name,opponent in opponents:
            jobs+=paired_tasks(cfg,namespace,difficulty,candidate,name,opponent,openings,mc)
    rows=run_jobs(jobs,cfg,time.time()+cfg['eval_max_hours']*3600)
    report={'version':VERSION,'genome_hash':digest(genome),'complete':len(rows)==len(jobs),'quick_test':cfg['QUICK_TEST'],
            'actual_stockfish_levels':levels,'completed_games':len(rows),'requested_games':len(jobs),'difficulties':{}}
    table=[]
    for difficulty in cfg['eval_difficulties']:
        subset=[r for r in rows if r['candidate']==difficulty]
        rating=elo_summary(subset,cfg['seed']); improvement=comparison([r for r in subset if r['opponent']=='default'])
        report['difficulties'][difficulty]={'rating':rating,'versus_default':improvement}
        for opponent in sorted(set(r['opponent'] for r in subset)):
            counts=comparison([r for r in subset if r['opponent']==opponent]); table.append({'difficulty':difficulty,'opponent':opponent,**counts})
        if cfg['QUICK_TEST']:
            print(f'Cristopher ({difficulty}): QUICK_TEST wiring only; no playing-strength or Elo claim.')
        elif rating['elo'] is None or rating['interval'] is None:
            print(f'Cristopher ({difficulty}) estimated Elo: insufficient evidence for a finite estimate and uncertainty.')
        else:
            error=max(rating['elo']-rating['interval'][0],rating['interval'][1]-rating['elo'])
            print(f"Cristopher ({difficulty}) estimated Elo ≈ {rating['elo']:.0f} ± {error:.0f} (approximate 95% interval)")
        print('Champion vs generation-0 default:',improvement)
    report['table']=table
    path=Path(cfg['root'])/'logs'/cfg['run_name']/namespace
    atomic_json(path/'summary.json',report)
    import csv
    with (path/'results.csv').open('w',newline='') as handle:
        if table:
            writer=csv.DictWriter(handle,fieldnames=list(table[0])); writer.writeheader();writer.writerows(table)
    print('Benchmark completion:',report['complete'],'| Truncated games are excluded from Elo.')
    print('Stockfish UCI_Elo calibration is not a FIDE rating. Improvement is a measured outcome, never guaranteed.')
    return report
'''
STAGE2_SOURCE = r'''
def warm_start_nn(genome,cfg):
    from tqdm.auto import trange
    rng=np.random.default_rng(cfg['seed']+9000)
    boards=[]; board=chess.Board()
    for i in trange(cfg['stage2_positions'],desc='NN warm-start positions',mininterval=2):
        if board.is_game_over(claim_draw=True) or board.ply()>160: board=chess.Board()
        board.push(list(board.legal_moves)[int(rng.integers(board.legal_moves.count()))]); boards.append(board.copy(stack=False))
    x=np.stack([nn_features(b) for b in boards]+[nn_features(b.mirror()) for b in boards])
    y=np.array([static_evaluate(b,genome)*(1 if b.turn else -1)/100 for b in boards],dtype=np.float32)
    y=np.concatenate((y,-y)); y=np.clip(y,-20,20)
    params=[rng.normal(0,.04,(768,64)).astype(np.float32),np.zeros(64,np.float32),
            rng.normal(0,.08,(64,32)).astype(np.float32),np.zeros(32,np.float32),
            rng.normal(0,.1,(32,1)).astype(np.float32),np.zeros(1,np.float32)]
    for _ in trange(cfg['stage2_warm_steps'],desc='Warm start from stage-1 evaluation',mininterval=2):
        ids=rng.integers(len(x),size=128); xb=x[ids]; target=y[ids,None]
        w1,b1,w2,b2,w3,b3=params
        h1=np.maximum(0,xb@w1+b1); h2=np.maximum(0,h1@w2+b2); pred=h2@w3+b3
        delta=np.clip(2*(pred-target)/len(ids),-1,1)
        d2=(delta@w3.T)*(h2>0); d1=(d2@w2.T)*(h1>0)
        grads=[xb.T@d1,d1.sum(0),h1.T@d2,d2.sum(0),h2.T@delta,delta.sum(0)]
        for p,g in zip(params,grads): p-=.01*np.clip(g,-5,5)
    return np.concatenate([p.ravel() for p in params]).astype(float)

def run_stage2(cfg,state):
    if not cfg['STAGE2']: print('Stage 2 disabled. Stage-1 genome remains the engine.'); return None
    if not state.get('champion'): raise RuntimeError('Complete a stage-1 generation first.')
    base=state['champion']['genome']; token=digest({'genome':base,'cfg':signature(cfg)})[:16]
    path=Path(cfg['root'])/'checkpoints'/cfg['run_name']/f'stage2_{token}.json'
    if path.exists(): saved=json.loads(path.read_text())
    else:
        weights=warm_start_nn(base,cfg)
        saved={'iteration':0,'mean':weights.tolist(),'rng':np.random.default_rng(cfg['seed']+19000).bit_generator.state}
        atomic_json(path,saved)
    deadline=time.time()+cfg['max_hours']*3600
    rng=np.random.default_rng(); rng.bit_generator.state=saved['rng']
    while saved['iteration']<cfg['stage2_iterations'] and time.time()<deadline:
        mean=np.array(saved['mean']); noise=rng.normal(size=(cfg['stage2_pairs'],len(mean)))
        perturbations=np.concatenate((noise,-noise)); sigma=cfg['stage2_sigma']
        samples=[{'genome':{'kind':'nn','weights':np.clip(mean+sigma*e,-5,5).tolist()}} for e in perturbations]
        fake={'generation':saved['iteration'],'population':samples,'hall':[{'genome':base}]}
        jobs=generation_tasks(cfg,fake)
        # Separate stage-2 artifacts from the stage-1 generations, including partial-game resume.
        for job in jobs:
            for key in ('pgn','result_path'): job[key]=job[key].replace(f"/g{saved['iteration']:05d}/",f"/stage2_{token}_{saved['iteration']:05d}/")
        rows=run_jobs(jobs,cfg,deadline)
        if len(rows)!=len(jobs): print('Stage 2 paused; rerun to reuse its completed matches.'); return None
        rewards=np.array([np.mean([r['fitness'] for r in rows if r['candidate']==i]) for i in range(len(samples))])
        centered=(rewards-rewards.mean())/(rewards.std()+1e-8)
        update=perturbations.T@centered/(len(samples)*sigma)
        mean=np.clip(mean+cfg['stage2_lr']*np.clip(update,-1,1),-5,5)
        saved={'iteration':saved['iteration']+1,'mean':mean.tolist(),'rng':rng.bit_generator.state}
        atomic_json(path,saved); print('ES iteration',saved['iteration'],'mean game fitness',float(rewards.mean()),flush=True)
    if saved['iteration']<cfg['stage2_iterations']: return None
    candidate={'kind':'nn','weights':saved['mean'],'stage1_genome_hash':digest(base)}
    openings=[random_opening(cfg['seed']+700000+i) for i in range(max(20,cfg['eval_games'])//2)]
    jobs=paired_tasks(cfg,f'stage2_gate_{token}',0,player('genome',candidate),'stage1',player('genome',base),openings,match_config(cfg,'hard'))
    rows=run_jobs(jobs,cfg,deadline); report=comparison(rows)
    # One-sided exact binomial test on independent, decisive opening-pair results.
    pairs={}
    for row in rows:
        if not row['truncated']: pairs.setdefault(row['pair'],[]).append(row['score'])
    pair_scores=[sum(v) for v in pairs.values() if len(v)==2]
    wins=sum(v>1 for v in pair_scores); losses=sum(v<1 for v in pair_scores); n=wins+losses
    pvalue=sum(math.comb(n,k) for k in range(wins,n+1))/2**n if n else 1.
    accepted=len(rows)==len(jobs) and len(pair_scores)>=10 and pvalue<.05 and wins>losses
    atomic_json(path.with_name(f'stage2_gate_{token}.json'),{'comparison':report,'paired_pvalue':pvalue,'accepted':accepted})
    print('Stage-2 promotion:',accepted,'paired p-value:',pvalue)
    return candidate if accepted else None
'''
WORKER_SOURCE='\n\n'.join((CORE_SOURCE,EVAL_SOURCE,SEARCH_SOURCE,CPP_WRAPPER_SOURCE,GA_SOURCE,REPORT_SOURCE,STAGE2_SOURCE))
worker_path=MODELS/'cristopher_evo_worker.py'
temp=worker_path.with_suffix('.tmp'); temp.write_text(WORKER_SOURCE); os.replace(temp,worker_path)
if str(MODELS) not in sys.path: sys.path.insert(0,str(MODELS))
importlib.invalidate_caches()
import cristopher_evo_worker as worker
worker=importlib.reload(worker)
print('Spawn-safe worker module:',worker_path)

GPU_SOURCE = r'''
# Included with CORE_SOURCE and EVAL_SOURCE; import torch only in the GPU-owning process.
import importlib, multiprocessing as mp, concurrent.futures, ctypes
GPU_VERSION = 'cristopher-gpu-es-1'
GPU_SEARCH_VERSION = 'shared-alpha-beta-v2'  # Existing weights remain compatible.
GPU_FEATURES = 784  # 781 chess features + 3 zeros for tensor-core alignment.
_GPU_NATIVE = {}

def gpu_features(board):
    x=np.zeros(GPU_FEATURES,np.float32); own=board.turn
    for square,piece in board.piece_map().items():
        oriented=square if own else square^56
        x[((0 if piece.color==own else 6)+piece.piece_type-1)*64+oriented]=1
    x[768:772]=[board.has_kingside_castling_rights(own),board.has_queenside_castling_rights(own),
                board.has_kingside_castling_rights(not own),board.has_queenside_castling_rights(not own)]
    if board.has_legal_en_passant(): x[772+chess.square_file(board.ep_square)]=1
    x[780]=min(100,board.halfmove_clock)/100
    return x

def gpu_terminal(board):
    if board.is_checkmate(): return -float(MATE)
    if board.is_stalemate() or board.is_insufficient_material() or board.is_fifty_moves() or board.is_repetition(3): return 0.
    return None

def gpu_prior(genome):
    prior=np.zeros(GPU_FEATURES,np.float32)
    for pt in range(1,7):
        material=genome[pt-1] if pt<6 else 0
        for sq in range(64):
            prior[(pt-1)*64+sq]=material+(genome[5+(pt-1)*64+sq]+genome[389+(pt-1)*64+sq])/2
            mirrored=sq^56
            prior[(pt+5)*64+sq]=-material-(genome[5+(pt-1)*64+mirrored]+genome[389+(pt-1)*64+mirrored])/2
    return prior

def gpu_native_library(path):
    if path not in _GPU_NATIVE:
        lib=ctypes.CDLL(path); fp=ctypes.POINTER(ctypes.c_float)
        lib.evo_children.argtypes=[ctypes.c_char_p,ctypes.c_char_p,ctypes.c_char_p,fp,fp,fp]
        lib.evo_children.restype=ctypes.c_int; _GPU_NATIVE[path]=lib
    return _GPU_NATIVE[path]

def gpu_prepare_one(item, native_path=''):
    # CPU workers own legal move generation/encoding. They never initialize CUDA.
    identifier,fen,moves=item
    if native_path:
        features=np.empty((256,GPU_FEATURES),np.float32); values=np.empty(256,np.float32)
        names=ctypes.create_string_buffer(256*6+1); terminal=ctypes.c_float()
        n=gpu_native_library(native_path).evo_children(fen.encode(),' '.join(moves).encode(),names,
            features.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),values.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),ctypes.byref(terminal))
        if n<0: raise RuntimeError('Native legal-move preparation failed')
        ucis=names.value.decode().split()
        return identifier,ucis,features[:n].astype(np.float16),values[:n].copy(),float(terminal.value)
    board=chess.Board(fen)
    for uci in moves: board.push_uci(uci)
    terminal=gpu_terminal(board)
    if terminal is not None:
        return identifier,[],np.empty((0,GPU_FEATURES),np.float16),np.empty(0,np.float32),terminal
    legal=sorted(board.legal_moves,key=lambda m:m.uci()); features=[]; values=[]
    for move in legal:
        board.push(move); value=gpu_terminal(board)
        features.append(gpu_features(board)); values.append(np.nan if value is None else value); board.pop()
    return identifier,[m.uci() for m in legal],np.asarray(features,np.float16),np.asarray(values,np.float32),float('nan')

def gpu_prepare_chunk(payload):
    items,path=payload
    return [gpu_prepare_one(item,path) for item in items]

def gpu_layout(cfg):
    sizes=[GPU_FEATURES,*cfg['gpu_hidden'],1]; layout=[]; offset=0
    for a,b in zip(sizes,sizes[1:]):
        layout.append((offset,a,b)); offset+=a*b+b
    return layout,offset

def gpu_initial_vector(cfg):
    import torch
    generator=torch.Generator().manual_seed(cfg['seed'])
    layout,count=gpu_layout(cfg); vector=torch.zeros(count); scale=torch.zeros(count)
    for i,(offset,a,b) in enumerate(layout):
        n=a*b; std=math.sqrt(2/a)
        if i<len(layout)-1: vector[offset:offset+n]=torch.randn(n,generator=generator)*std
        scale[offset:offset+n]=1/math.sqrt(a); scale[offset+n:offset+n+b]=.1
    return vector,scale

class GpuPopulation:
    def __init__(self,cfg,state,device=None):
        import torch
        self.torch=torch; self.cfg=cfg
        self.device=torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        if self.device.type!='cuda' and cfg['gpu_require_cuda']:
            raise RuntimeError('GPU neuroevolution requires a Colab GPU runtime. Select a GPU, reconnect, and rerun sections 0–5. No CPU-only training was started.')
        self.dtype=torch.float16 if self.device.type=='cuda' else torch.float32
        self.layout,self.count=gpu_layout(cfg)
        self.mean=state['mean'].to(self.device)
        self.scale=None if cfg.get('gpu_inference_only') else gpu_initial_vector(cfg)[1].to(self.device)
        self.seeds=state['seeds']
        if self.seeds and state.get('noise_device',self.device.type)!=self.device.type:
            raise RuntimeError('Resume ES perturbations on the same device family (CUDA or CPU) used by this run.')
        self.sigma=state['sigma']; self.pairs=cfg['gpu_population']//2
        vectors=[]; self.noise=[]
        with torch.inference_mode():
            for seed in self.seeds:
                gen=torch.Generator(device=self.device).manual_seed(seed)
                noise=torch.randn(self.count,generator=gen,device=self.device)
                self.noise.append(noise)
                vectors.extend((self.mean+self.sigma*self.scale*noise,self.mean-self.sigma*self.scale*noise))
            self.mean_id=len(vectors); vectors.append(self.mean)
            self.hall_ids=[]
            if cfg.get('gpu_inference_only'):
                self.champion_id=self.default_id=self.mean_id
            else:
                self.champion_id=len(vectors); vectors.append(state['champion'].to(self.device))
                self.default_id=len(vectors); vectors.append(state['initial'].to(self.device))
                for weights in state['hall']:
                    self.hall_ids.append(len(vectors)); vectors.append(weights.to(self.device))
            stacked=torch.stack(vectors).to(self.dtype)
            self.layers=[]
            for offset,a,b in self.layout:
                self.layers.append((stacked[:,offset:offset+a*b].reshape(-1,a,b).contiguous(),
                                    stacked[:,offset+a*b:offset+a*b+b].reshape(-1,1,b).contiguous()))
            del vectors,stacked
        self.prior=torch.as_tensor(state['prior'],device=self.device,dtype=self.dtype)
        self.batch_rows=cfg['gpu_batch_rows']; self.positions=0; self.forward_seconds=0.
        self.budget=None
        if self.device.type=='cuda':
            free,total=torch.cuda.mem_get_info(self.device)
            available=free+torch.cuda.memory_reserved(self.device)-cfg['gpu_headroom_gb']*2**30
            self.budget=min(cfg['gpu_target_vram_gb']*2**30,available)
            if self.budget<=torch.cuda.memory_allocated(self.device):
                raise RuntimeError('GPU model population does not fit with the requested headroom. Reduce gpu_population or gpu_hidden for a new run.')
            torch.cuda.reset_peak_memory_stats(self.device)
        print(f'Neural population device: {self.device} | {len(self.layers[0][0])} networks | {self.count:,} parameters each',flush=True)
    def forward(self,x,ids):
        torch=self.torch
        base=(x*self.prior).sum(-1).float()+10.
        for i,(weight,bias) in enumerate(self.layers):
            x=torch.bmm(x,weight.index_select(0,ids))+bias.index_select(0,ids)
            if i<len(self.layers)-1: x=torch.relu(x)
        return base+self.cfg['gpu_residual_cp']*torch.tanh(x.squeeze(-1).float())
    def evaluate(self,requests):
        # Group positions by candidate: actual batched GEMMs, no per-position CUDA calls.
        torch=self.torch; result={}; groups={}
        for key,model,features in requests:
            groups.setdefault(model,[]).append((key,features))
        if not groups: return result
        merged={}; offsets={}
        for model,items in groups.items():
            merged[model]=np.concatenate([x for _,x in items]); cursor=0; offsets[model]=[]
            for key,features in items:
                offsets[model].append((key,cursor,len(features))); cursor+=len(features)
        predictions={model:np.empty(len(features),np.float32) for model,features in merged.items()}
        start=time.perf_counter(); pending={m:0 for m in merged}
        with torch.inference_mode():
            while pending:
                # Bucket by remaining row count to avoid padding small candidate batches
                # to the size of a shared champion/opponent's much larger batch.
                def bucket(model):
                    remaining=len(merged[model])-pending[model]
                    return min(self.batch_rows,1<<(remaining-1).bit_length())
                selected=min(bucket(m) for m in pending)
                models=[m for m in pending if bucket(m)==selected]
                size=min(selected,max(len(merged[m])-pending[m] for m in models))
                host=np.zeros((len(models),size,GPU_FEATURES),np.float16 if self.dtype==torch.float16 else np.float32)
                for i,model in enumerate(models):
                    offset=pending[model]; data=merged[model][offset:offset+size]; host[i,:len(data)]=data
                x=ids=values=None
                try:
                    tensor=torch.from_numpy(host)
                    if self.device.type=='cuda': tensor=tensor.pin_memory()
                    x=tensor.to(self.device,non_blocking=True); ids=torch.tensor(models,device=self.device)
                    values=self.forward(x,ids)
                    if not torch.isfinite(values).all(): raise FloatingPointError('Non-finite neural values; checkpoint preserved.')
                    if self.budget and torch.cuda.memory_reserved(self.device)>self.budget and size>1:
                        del tensor,x,ids,values; x=ids=values=None
                        self.batch_rows=max(1,size//2); torch.cuda.empty_cache(); continue
                    cpu=values.cpu().numpy()
                    for i,model in enumerate(models):
                        offset=pending[model]; length=min(size,len(merged[model])-offset)
                        predictions[model][offset:offset+length]=cpu[i,:length]
                        pending[model]+=length
                        if pending[model]==len(merged[model]): del pending[model]
                except torch.cuda.OutOfMemoryError:
                    x=ids=values=None
                    if size<=1: raise
                    self.batch_rows=max(1,size//2); torch.cuda.empty_cache()
                    print('Reduced GPU chunk after OOM to',self.batch_rows,'rows per network',flush=True)
                finally:
                    x=ids=values=None
        for model,items in offsets.items():
            for key,start_index,length in items: result[key]=predictions[model][start_index:start_index+length]
        self.positions+=sum(map(len,merged.values())); self.forward_seconds+=time.perf_counter()-start
        return result
    def candidate(self,index):
        if index==self.mean_id: return self.mean.detach().cpu().clone()
        noise=self.noise[index//2]; sign=1 if index%2==0 else -1
        return (self.mean+sign*self.sigma*self.scale*noise).detach().cpu()
    def update(self,fitness):
        torch=self.torch; values=np.asarray(fitness[:self.cfg['gpu_population']],float)
        # Tied fitness gets tied utility: draws alone cannot generate arbitrary evolution.
        utilities=np.zeros(len(values))
        for value in np.unique(values):
            equal=values==value
            utilities[equal]=(np.sum(values<value)+(np.sum(equal)-1)/2)/max(1,len(values)-1)-.5
        gradient=torch.zeros_like(self.mean)
        for i,noise in enumerate(self.noise): gradient.add_(noise,alpha=float(utilities[2*i]-utilities[2*i+1]))
        gradient/=max(1,self.pairs*self.sigma)
        next_mean=self.mean+self.cfg['gpu_learning_rate']*self.scale*gradient.clamp(-1,1)
        sigma=float(np.clip(self.sigma*(1.05 if values.std()<.02 else .995),.02,.5))
        return next_mean.detach().cpu(),sigma
    def status(self):
        if self.device.type!='cuda': return 'CPU test mode (not accelerated)'
        t=self.torch
        return f'CUDA allocated {t.cuda.memory_allocated()/2**30:.2f} GiB, reserved {t.cuda.memory_reserved()/2**30:.2f} GiB, peak {t.cuda.max_memory_allocated()/2**30:.2f} GiB'

def gpu_signature(cfg):
    keys=('gpu_hidden','gpu_population','gpu_pairs_per_opponent','gpu_sigma','gpu_learning_rate',
          'gpu_residual_cp','gpu_temperature','gpu_max_plies','seed','gpu_seed_classic_run',
          'gpu_anchor_every','gpu_anchor_enabled','anchor_elo','stockfish_seconds')
    return digest({'version':GPU_VERSION,**{key:cfg[key] for key in keys}})

def gpu_save(cfg,state):
    import torch
    folder=Path(cfg['root'])/'checkpoints'/cfg['run_name']; folder.mkdir(parents=True,exist_ok=True)
    state['sequence']+=1; path=folder/f"gpu_{state['sequence']:08d}.pt"
    temporary=path.with_suffix('.tmp')
    with temporary.open('wb') as handle:
        torch.save({'signature':gpu_signature(cfg),'CFG':cfg,'state':state},handle)
        handle.flush(); os.fsync(handle.fileno())
    os.replace(temporary,path)
    gpu_export(cfg,state,Path(__file__).read_text())
    for old in sorted(folder.glob('gpu_*.pt'))[:-3]: old.unlink()
    return path

def gpu_load(cfg,create=False):
    import torch
    folder=Path(cfg['root'])/'checkpoints'/cfg['run_name']
    paths=sorted(folder.glob('gpu_*.pt'),reverse=True)
    for path in paths:
        try: pack=torch.load(path,map_location='cpu',weights_only=True)
        except Exception as error: print('Unreadable checkpoint:',path.name,type(error).__name__); continue
        if pack['signature']!=gpu_signature(cfg): raise ValueError('GPU experiment settings changed. Restore them or select a new run_name.')
        print('GPU resume:',path.name,'generation',pack['state']['generation'],flush=True); return pack['state']
    if paths: raise RuntimeError('No readable GPU checkpoint remains; refusing to restart silently.')
    if not create: raise RuntimeError('No GPU run saved yet. Run train.ipynb first.')
    initial,_=gpu_initial_vector(cfg); genome=genome_default(); source='handcrafted material/PST prior'
    classic=Path(cfg['root'])/'checkpoints'/cfg['gpu_seed_classic_run']/'best_champion.json'
    if cfg.get('gpu_seed_classic_run') and classic.exists():
        genome=np.asarray(json.loads(classic.read_text())['genome']); source=str(classic)
    rng=np.random.default_rng(cfg['seed'])
    state={'generation':0,'sequence':0,'mean':initial.clone(),'initial':initial,'champion':initial.clone(),
           'champion_generation':-1,'sigma':cfg['gpu_sigma'],'prior':gpu_prior(genome).tolist(),'prior_source':source,
           'hall':[],'jobs':[],'seeds':[], 'noise_device':'cuda' if torch.cuda.is_available() else 'cpu', 'rng':rng.bit_generator.state,'history':[], 'positions':0,'elapsed_seconds':0.}
    gpu_next_seeds(state,cfg); gpu_save(cfg,state); return state

def gpu_next_seeds(state,cfg):
    rng=np.random.default_rng(); rng.bit_generator.state=state['rng']
    state['seeds']=[int(x) for x in rng.integers(0,2**31-1,size=cfg['gpu_population']//2)]
    state['rng']=rng.bit_generator.state

def gpu_jobs(cfg,state,bank):
    jobs=[]; generation=state['generation']
    for candidate in range(cfg['gpu_population']+1):
        peer=(candidate+generation+1)%cfg['gpu_population']
        if peer==candidate: peer=(peer+1)%cfg['gpu_population']
        opponents=[('js',-1),('random',-1),('default',bank.default_id),('peer',peer),('champion',bank.champion_id)]
        if bank.hall_ids: opponents.append(('hall',bank.hall_ids[generation%len(bank.hall_ids)]))
        if cfg['gpu_anchor_enabled'] and generation%cfg['gpu_anchor_every']==0: opponents.append(('stockfish',-2))
        for name,opponent in opponents:
            for pair in range(cfg['gpu_pairs_per_opponent']):
                opening=random_opening(cfg['seed']+generation*1009+pair)
                for candidate_white in (True,False):
                    jobs.append({'id':len(jobs),'candidate':candidate,'opponent':name,'pair':pair,
                                 'white':candidate if candidate_white else opponent,'black':opponent if candidate_white else candidate,
                                 'candidate_white':candidate_white,'moves':list(opening),'done':False})
    return jobs

def gpu_result_path(cfg,state,job):
    return Path(cfg['root'])/'logs'/cfg['run_name']/f"gpu_g{state['generation']:06d}"/f"game_{job['id']:05d}.json"

def gpu_finish(cfg,state,job,value=None,truncated=False):
    white_turn=len(job['moves'])%2==0
    winner=None if value==0 or truncated else (white_turn if value>0 else not white_turn)
    score=.5 if winner is None else float(winner==job['candidate_white'])
    job.update(done=True,score=score,truncated=truncated,
               fitness=score+(.01*(1-len(job['moves'])/cfg['gpu_max_plies']) if score==1 else 0))
    board=chess.Board()
    for move in job['moves']: board.push_uci(move)
    pgn=chess.pgn.Game.from_board(board)
    pgn.headers['Result']='*' if truncated else '1/2-1/2' if winner is None else '1-0' if winner else '0-1'
    pgn.headers['Event']='Cristopher CUDA neuroevolution'
    pgn.headers['Termination']='ply cap: unfinished' if truncated else 'terminal position'
    path=Path(cfg['root'])/'pgn'/cfg['run_name']/f"gpu_g{state['generation']:06d}"/f"game_{job['id']:05d}.pgn"
    path.parent.mkdir(parents=True,exist_ok=True); temp=path.with_suffix('.tmp'); temp.write_text(str(pgn)+'\n\n'); os.replace(temp,path)
    atomic_json(gpu_result_path(cfg,state,job),job)

def gpu_choose(values,cfg,generation,job):
    best=int(np.argmax(values))
    if values[best]>MATE-1000: return best
    if len(job['moves'])>=24 or cfg['gpu_temperature']<=0: return best
    rng=np.random.default_rng(np.random.SeedSequence([cfg['seed'],generation,job['id'],len(job['moves'])]))
    logits=(values-values.max())/cfg['gpu_temperature']; probabilities=np.exp(np.clip(logits,-60,0)); probabilities/=probabilities.sum()
    return int(rng.choice(len(values),p=probabilities))

def gpu_evolve(cfg):
    import torch,gc
    if cfg['gpu_require_cuda'] and not torch.cuda.is_available():
        raise RuntimeError('Select a GPU runtime in Colab first. This cell will not silently train on CPU.')
    torch.set_num_threads(1)
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32=True
        print('Training GPU:',torch.cuda.get_device_name(0),flush=True)
    state=gpu_load(cfg,create=True)
    gpu_export(cfg,state,Path(__file__).read_text())
    began=time.monotonic(); deadline=began+cfg['max_hours']*3600
    last_save=began; last_report=began; bank=None; pool=None; sf=None
    def save():
        nonlocal last_save
        now=time.monotonic(); state['elapsed_seconds']+=now-last_save
        gpu_save(cfg,state); last_save=now
    def schedule(ids):
        items=[(i,chess.STARTING_FEN,list(state['jobs'][i]['moves'])) for i in ids]
        chunks=[items[i:i+cfg['gpu_cpu_chunk']] for i in range(0,len(items),cfg['gpu_cpu_chunk'])]
        payloads=[(chunk,cfg.get('gpu_native_path','')) for chunk in chunks]
        if pool is None: return [gpu_prepare_chunk(payload) for payload in payloads]
        return [pool.submit(gpu_prepare_chunk,payload) for payload in payloads]
    def collect(pending):
        return [row for item in pending for row in (item.result() if pool is not None else item)]
    try:
        if cfg['workers']>1 and not cfg.get('gpu_train_with_search'):
            pool=concurrent.futures.ProcessPoolExecutor(max_workers=cfg['workers'],mp_context=mp.get_context('spawn'))
        while state['generation']<cfg['gpu_generations'] and time.monotonic()<deadline:
            bank=GpuPopulation(cfg,state)
            if not state['jobs']: state['jobs']=gpu_jobs(cfg,state,bank); save()
            for i,job in enumerate(state['jobs']):
                path=gpu_result_path(cfg,state,job)
                if not job['done'] and path.exists(): state['jobs'][i]=json.loads(path.read_text())
            print(f"Generation {state['generation']} | {len(state['jobs'])} paired games | CPU workers {cfg['workers']} | {bank.status()}",flush=True)
            if cfg.get('gpu_train_with_search'):
                _CHESS_ENGINE_PACKAGE.notebook_bridge.search_training_games(__import__('sys').modules[__name__],cfg,state,bank,deadline,save)
            pending=None; current=[]
            while not cfg.get('gpu_train_with_search') and any(not job['done'] for job in state['jobs']) and time.monotonic()<deadline:
                active=[job['id'] for job in state['jobs'] if not job['done']]
                if pending is None:
                    width=min(cfg['gpu_parallel_games'],max(1,(len(active)+1)//2))
                    current=active[:width]; pending=schedule(current)
                prepared=collect(pending); pending=None
                # Prepare a disjoint group while the GPU evaluates this group.
                excluded=set(current); other=[i for i in active if i not in excluded][:cfg['gpu_parallel_games']]
                following=schedule(other) if other else None
                requests=[]; acting={}; ready=[]
                for identifier,moves,features,terminals,root_terminal in prepared:
                    job=state['jobs'][identifier]
                    if np.isfinite(root_terminal): gpu_finish(cfg,state,job,root_terminal); continue
                    if len(job['moves'])>=cfg['gpu_max_plies']: gpu_finish(cfg,state,job,truncated=True); continue
                    if not moves: raise RuntimeError('Nonterminal board had no legal move')
                    model=job['white'] if len(job['moves'])%2==0 else job['black']
                    acting[identifier]=model; ready.append((identifier,moves,terminals))
                    if model>=0: requests.append((identifier,model,features))
                estimates=bank.evaluate(requests)
                state['positions']+=sum(len(features) for _,_,features in requests)
                for identifier,moves,terminals in ready:
                    job=state['jobs'][identifier]
                    if time.monotonic()>=deadline: break
                    if acting[identifier]==-2:
                        if sf is None:
                            sf=chess.engine.SimpleEngine.popen_uci(cfg['stockfish'],timeout=10)
                            option=sf.options['UCI_Elo']
                            sf.configure({'Threads':1,'Hash':16,'UCI_LimitStrength':True,'UCI_Elo':max(option.min,min(option.max,cfg['anchor_elo']))})
                        position=chess.Board()
                        for uci in job['moves']: position.push_uci(uci)
                        answer=sf.play(position,chess.engine.Limit(time=cfg['stockfish_seconds']),game=(state['generation'],identifier))
                        choice=moves.index(answer.move.uci())
                    elif acting[identifier]<0:
                        rng=np.random.default_rng(np.random.SeedSequence([cfg['seed'],state['generation'],identifier,len(job['moves'])]))
                        choice=int(rng.integers(len(moves)))
                    else:
                        values=-estimates[identifier]; mask=np.isfinite(terminals); values[mask]=-terminals[mask]
                        choice=gpu_choose(values,cfg,state['generation'],job)
                    job['moves'].append(moves[choice])
                    if np.isfinite(terminals[choice]): gpu_finish(cfg,state,job,float(terminals[choice]))
                pending=following; current=other
                now=time.monotonic()
                if now-last_save>=cfg['gpu_checkpoint_seconds']: save()
                if now-last_report>=cfg['gpu_progress_seconds']:
                    done=sum(job['done'] for job in state['jobs'])
                    print(f"GPU TRAINING | generation {state['generation']} | games {done}/{len(state['jobs'])} | neural positions {state['positions']:,} | {bank.positions/max(bank.forward_seconds,1e-9):,.0f} positions/s in inference | {bank.status()} | {max(0,deadline-now)/60:.1f} min left",flush=True)
                    last_report=now
            if any(not job['done'] for job in state['jobs']): break
            fitness=[float(np.mean([j['fitness'] for j in state['jobs'] if j['candidate']==i])) for i in range(cfg['gpu_population']+1)]
            winner=int(np.argmax(fitness)); candidate=bank.candidate(winner)
            next_mean,sigma=bank.update(fitness)
            # Promote only a contender that actually beat the frozen champion, not capped draws.
            gate=[j for j in state['jobs'] if j['candidate']==winner and j['opponent']=='champion']
            qualified=bool(gate and all(not j['truncated'] for j in gate) and np.mean([j['score'] for j in gate])>.5)
            promoted=False
            if qualified:
                print('Independent champion validation:',state['generation'],flush=True)
                save()
                promoted=_CHESS_ENGINE_PACKAGE.notebook_bridge.validate_neural(__import__('sys').modules[__name__],cfg,state,bank,winner,deadline)
                if promoted is None: break  # Match files and this generation resume next session.
            champion=candidate if promoted else state['champion']
            hall=(state['hall']+[state['champion'].clone()])[-cfg['gpu_hall_size']:] if promoted else state['hall']
            record={'generation':state['generation'],'best':max(fitness),'mean':float(np.mean(fitness)),
                    'sigma':sigma,'promoted':promoted,'truncated':sum(j['truncated'] for j in state['jobs'])/len(state['jobs']),
                    'positions':state['positions'],'cuda':bank.device.type=='cuda'}
            state={**state,'mean':next_mean,'sigma':sigma,'champion':champion,'hall':hall,'jobs':[],
                   'champion_generation':state['generation'] if promoted else state['champion_generation'],
                   'generation':state['generation']+1,'history':state['history']+[record]}
            gpu_next_seeds(state,cfg); save(); print('Evolution update:',record,flush=True)
            atomic_json(Path(cfg['root'])/'logs'/cfg['run_name']/'gpu_history.json',state['history'])
            bank=None; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    except KeyboardInterrupt:
        print('Interrupted. Saving model population and in-progress games.',flush=True)
    finally:
        save()
        if pool is not None: pool.shutdown(wait=True,cancel_futures=True)
        if sf is not None: sf.quit()
    print('Saved GPU training session. Run all in train.ipynb to resume.',flush=True)
    return state

class GpuCristopherEngine:
    def __init__(self,model_path,seed=42):
        import torch
        self.torch=torch; path=Path(model_path); metadata=json.loads(path.read_text())
        if metadata.get('version')!=GPU_VERSION: raise ValueError('Not a GPU-neuroevolution export')
        pack=torch.load(path.parent/metadata['weights'],map_location='cpu',weights_only=True)
        self.cfg=metadata['CFG']; self.settings=metadata['difficulty']; self.rng=np.random.default_rng(seed)
        # A single network uses the same grouped CUDA kernels; CPU inference is also supported.
        cfg={**self.cfg,'gpu_population':0,'gpu_require_cuda':False,'gpu_batch_rows':4096,'gpu_inference_only':True}
        state={'mean':pack['weights'],'champion':pack['weights'],'initial':pack['weights'],
               'prior':pack['prior'],'seeds':[],'sigma':0.,'hall':[]}
        self.bank=GpuPopulation(cfg,state); self.prior_cpu=np.asarray(pack['prior'],np.float32); self.book=metadata.get('opening_book',{})
    def values(self,boards):
        if not boards: return np.empty(0,np.float32)
        features=np.stack([gpu_features(b) for b in boards]).astype(np.float16)
        values=self.bank.evaluate([(0,0,features)])[0]
        for i,board in enumerate(boards):
            terminal=gpu_terminal(board)
            if terminal is not None: values[i]=terminal
        return values
    def beam_children(self,board,beam,deadline):
        """Keep terminal outcomes and every check evasion before heuristic pruning.

        The beam limits only nonterminal continuations outside check. Return None
        on interruption so lookahead keeps its last fully completed iteration.
        """
        checked=board.is_check(); protected=[]; candidates=[]
        for move in board.legal_moves:
            if time.monotonic()>=deadline: return None
            child=board.copy(stack=True); child.push(move)
            if checked or gpu_terminal(child) is not None:
                protected.append(child)
            else:
                candidates.append(child)
        if len(candidates)>beam:
            candidates.sort(key=lambda b: -(gpu_features(b)@self.prior_cpu+10),reverse=True)
            candidates=candidates[:beam]
        return protected+candidates
    def lookahead(self,board,max_depth,beam,deadline):
        moves=sorted(board.legal_moves,key=lambda m:m.uci()); children=[]
        for move in moves:
            child=board.copy(stack=True);child.push(move);children.append(child)
        scores=-self.values(children); best=int(np.argmax(scores)); completed=1; nodes=len(children)
        if max_depth==1: return moves,scores,completed,nodes
        # Breadth expansion allows one neural batch for many independent search nodes.
        for target_depth in range(2,max_depth+1):
            layers=[children]; links=[]; aborted=False
            for level in range(1,target_depth):
                parents=layers[-1]; next_layer=[]; mapping=[]
                for parent in parents:
                    if time.monotonic()>=deadline: aborted=True;break
                    if gpu_terminal(parent) is not None: mapping.append([]);continue
                    options=self.beam_children(parent,beam,deadline)
                    if options is None: aborted=True;break
                    mapping.append(list(range(len(next_layer),len(next_layer)+len(options))));next_layer.extend(options)
                if aborted: break
                links.append(mapping);layers.append(next_layer)
            if aborted or time.monotonic()>=deadline: break
            values=self.values(layers[-1]); nodes+=len(values)
            for level in range(len(links)-1,-1,-1):
                reduced=[]
                for parent,indices in zip(layers[level],links[level]):
                    terminal=gpu_terminal(parent)
                    reduced.append(terminal if terminal is not None else -float(np.min(values[indices])))
                values=np.asarray(reduced,np.float32)
            scores=-values; completed=target_depth
        return moves,scores,completed,nodes
    def get_move(self,fen: str,history: list[str] | None=None,difficulty: str='medium') -> dict:
        started=time.monotonic();board=board_from_history(fen,history)
        if difficulty not in self.settings: raise ValueError('Choose easy, medium, or hard')
        legal=list(board.legal_moves)
        if not legal: raise ValueError('Game over: no legal moves')
        for move in legal:
            board.push(move);mate=board.is_checkmate();board.pop()
            if mate: return {'move':move.uci(),'info':{'eval':float(MATE-1),'depth':1,'nodes':0,'time_ms':int(1000*(time.monotonic()-started))}}
        limits=self.settings[difficulty]
        if difficulty != 'easy' and self.cfg.get('gpu_search_backend','alpha_beta') == 'alpha_beta':
            return _CHESS_ENGINE_PACKAGE.notebook_bridge.neural_move(self,board,limits)
        deadline=started+limits['seconds']
        max_depth={'easy':1,'medium':2,'hard':3}[difficulty];beam=4 if difficulty=='medium' else 8
        moves,scores,depth,nodes=self.lookahead(board,max_depth,beam,deadline)
        if history and self.values([board])[0]>50:
            repeated=[]
            for move in moves:
                board.push(move);repeated.append(board.is_repetition(3));board.pop()
            if not all(repeated): scores[np.asarray(repeated)]=-MATE
        index=int(np.argmax(scores))
        if difficulty=='easy':
            indices=np.arange(len(moves)) if self.rng.random()<limits.get('inaccuracy',.12) else np.argsort(scores)[-5:]; logits=scores[indices]/max(1,limits['temperature'])
            probabilities=np.exp(logits-logits.max());probabilities/=probabilities.sum()
            index=int(self.rng.choice(indices,p=probabilities))
        move=moves[index]; assert move in board.legal_moves
        return {'move':move.uci(),'info':{'eval':float(scores[index]),'depth':int(depth),'nodes':int(nodes),'time_ms':int(1000*(time.monotonic()-started))}}

def gpu_export(cfg,state,source):
    artifact={'version':GPU_VERSION,'search_version':GPU_SEARCH_VERSION,'CFG':cfg,'difficulty':cfg['difficulty'],
              'generation':state['generation'],'champion_generation':state['champion_generation'],
              'opening_book':JS_OPENING_BOOK,'method':'antithetic evolution strategies; reward from played games only',
              'promotion_protocol':state.get('promotion_protocol','legacy-unverified'),
              'champion_verified':state.get('champion_verified',False),
              'champion_validation':state.get('champion_validation'),
              'note':'Promotion requires independent paired matches. Earlier champions are retained as incumbents.'}
    source=source+'\nCristopherEngine = GpuCristopherEngine\n'
    return publish_champion(cfg,artifact,source,
                            {'weights':state['champion'].detach().cpu(),'prior':state['prior']})

def gpu_evaluate(cfg,state):
    import torch
    from tqdm.auto import tqdm
    identity=hashlib.sha256(state['champion'].numpy().tobytes()).hexdigest()
    folder=Path(cfg['root'])/'logs'/cfg['run_name']/('gpu_evaluation_'+digest({'weights':identity,'difficulty':cfg['difficulty'],'games':cfg['eval_games'],'levels':cfg['eval_levels']})[:16])
    folder.mkdir(parents=True,exist_ok=True)
    def engine_for(name,weights):
        torch.save({'weights':weights,'prior':state['prior']},folder/(name+'.pt'))
        atomic_json(folder/(name+'.json'),{'version':GPU_VERSION,'weights':name+'.pt','CFG':cfg,'difficulty':cfg['difficulty']})
        return GpuCristopherEngine(folder/(name+'.json'),seed=cfg['seed'])
    champion=engine_for('champion',state['champion']); default=engine_for('default',state['initial'])
    deadline=time.monotonic()+cfg['eval_max_hours']*3600
    rows=[]
    with chess.engine.SimpleEngine.popen_uci(cfg['stockfish'],timeout=10) as sf:
        option=sf.options['UCI_Elo'];levels=sorted(set(max(option.min,min(option.max,int(x))) for x in cfg['eval_levels']))
        opponents=[('js',None),('random',None),('default',None)]+[(f'stockfish_{x}',x) for x in levels]
        jobs=[(difficulty,name,elo,i) for difficulty in cfg['eval_difficulties'] for name,elo in opponents for i in range(cfg['eval_games'])]
        for difficulty,name,elo,i in tqdm(jobs,desc='GPU champion evaluation',mininterval=2):
            path=folder/f'{difficulty}_{name}_{i:03d}.json'
            if path.exists():rows.append(json.loads(path.read_text()));continue
            if time.monotonic()>=deadline:break
            board=chess.Board();opening=random_opening(cfg['seed']+800000+i//2)
            for move in opening:board.push_uci(move)
            color=i%2==0; rng=random.Random(cfg['seed']+i);champion.rng=np.random.default_rng(cfg['seed']+i)
            default.rng=np.random.default_rng(cfg['seed']+i+1)
            if elo is not None:sf.configure({'Threads':1,'Hash':16,'UCI_LimitStrength':True,'UCI_Elo':elo})
            while gpu_terminal(board) is None and board.ply()<cfg['max_plies']:
                if time.monotonic()>=deadline:break
                if board.turn==color:
                    uci=champion.get_move(board.fen(),[m.uci() for m in board.move_stack],difficulty)['move']
                    move=board.parse_uci(uci)
                elif name in ('js','random'):move=rng.choice(list(board.legal_moves))
                elif name=='default':move=board.parse_uci(default.get_move(board.fen(),[m.uci() for m in board.move_stack],difficulty)['move'])
                else:move=sf.play(board,chess.engine.Limit(time=cfg['stockfish_seconds']),game=path.name).move
                board.push(move)
            if time.monotonic()>=deadline:break
            terminal=gpu_terminal(board);truncated=terminal is None
            winner=None if terminal==0 or truncated else (board.turn if terminal>0 else not board.turn)
            score=.5 if winner is None else float(winner==color)
            row={'candidate':difficulty,'opponent':name,'elo':elo,'pair':i//2,'candidate_white':color,
                 'score':score,'truncated':truncated,'plies':board.ply()}
            game=chess.pgn.Game.from_board(board)
            game.headers['Result']='*' if truncated else '1/2-1/2' if winner is None else '1-0' if winner else '0-1'
            pgn=Path(cfg['root'])/'pgn'/cfg['run_name']/folder.name/(path.stem+'.pgn');pgn.parent.mkdir(parents=True,exist_ok=True);pgn.write_text(str(game)+'\n\n')
            atomic_json(path,row);rows.append(row)
    report={'mode':GPU_VERSION,'weight_hash':identity,'complete':len(rows)==len(jobs),'games':len(rows),'table':[]}
    for difficulty in cfg['eval_difficulties']:
        subset=[r for r in rows if r['candidate']==difficulty];rating=elo_summary(subset,cfg['seed'])
        for name,_ in opponents:report['table'].append({'difficulty':difficulty,'opponent':name,**comparison([r for r in subset if r['opponent']==name])})
        report[difficulty]=rating
        if rating['elo'] is not None and rating['interval'] is not None:
            error=max(rating['elo']-rating['interval'][0],rating['interval'][1]-rating['elo'])
            print(f"Cristopher ({difficulty}) estimated Elo ≈ {rating['elo']:.0f} ± {error:.0f} (approximate engine calibration)")
        else:print(f'Cristopher ({difficulty}): insufficient evidence for an Elo estimate.')
    atomic_json(folder/'summary.json',report)
    return report

# Single-model inference has its own allocation policy; ES training is unchanged.
GpuPopulation = _CHESS_ENGINE_PACKAGE.inference.population_class(GpuPopulation, gpu_layout)
'''

GPU_NATIVE_SOURCE = r'''
#include "chess.hpp"
#include <cstring>
#include <cmath>
#include <algorithm>
#include <sstream>
#include <vector>
using namespace chess;
static float terminal(const Board& b) {
 auto result=b.isGameOver().second;
 if(result==GameResult::NONE)return NAN;
 if(result==GameResult::DRAW)return 0;
 return result==GameResult::WIN?30000.f:-30000.f;
}
static void encode(const Board&b,float*x){
 std::fill(x,x+784,0.f);auto own=b.sideToMove();
 for(int s=0;s<64;s++){
  auto p=b.at(Square(s));if(p==Piece::NONE)continue;
  int oriented=own==Color::WHITE?s:s^56;
  x[((p.color()==own?0:6)+int(p.type()))*64+oriented]=1;
 }
 auto rights=b.castlingRights();using Side=Board::CastlingRights::Side;
 x[768]=rights.has(own,Side::KING_SIDE);x[769]=rights.has(own,Side::QUEEN_SIDE);
 x[770]=rights.has(~own,Side::KING_SIDE);x[771]=rights.has(~own,Side::QUEEN_SIDE);
 if(b.enpassantSq()!=Square::NO_SQ){
  Movelist moves;movegen::legalmoves<movegen::MoveGenType::CAPTURE>(moves,b);
  for(auto move:moves)if(move.typeOf()==Move::ENPASSANT){x[772+b.enpassantSq().index()%8]=1;break;}
 }
 x[780]=std::min(100u,b.halfMoveClock())/100.f;
}
extern "C" int evo_children(const char* fen,const char* history,char*names,float*features,float*values,float*root_terminal){
 try {
  Board b(fen);std::istringstream stream(history);std::string u;
  while(stream>>u){auto m=uci::uciToMove(b,u);b.makeMove<true>(m);}
  *root_terminal=terminal(b);names[0]=0;
  if(!std::isnan(*root_terminal))return 0;
  Movelist legal;movegen::legalmoves(legal,b);if(legal.size()>256)return -1;
  std::vector<std::pair<std::string,Move>> moves;
  for(auto m:legal)moves.emplace_back(uci::moveToUci(m),m);
  std::sort(moves.begin(),moves.end(),[](const auto&a,const auto&z){return a.first<z.first;});
  std::string text;int i=0;
  for(auto entry:moves){
   text+=entry.first+" ";b.makeMove<true>(entry.second);
   encode(b,features+i*784);values[i]=terminal(b);b.unmakeMove(entry.second);i++;
  }
  std::strcpy(names,text.c_str());return i;
 }catch(...){return -1;}
}
'''


if CFG['training_mode'] in ('gpu_neuroevolution','parameter_evolution','full_pipeline'):
    GPU_WORKER_SOURCE='\n\n'.join((CORE_SOURCE,EVAL_SOURCE,REPORT_SOURCE,GPU_SOURCE))
    gpu_path=MODELS/'cristopher_gpu_worker.py'
    temporary=gpu_path.with_suffix('.tmp');temporary.write_text(GPU_WORKER_SOURCE);os.replace(temporary,gpu_path)
    if str(MODELS) not in sys.path:sys.path.insert(0,str(MODELS))
    importlib.invalidate_caches()
    import cristopher_gpu_worker as gpu_worker
    gpu_worker=importlib.reload(gpu_worker)
    CFG['gpu_native_path']=''
    if CFG['gpu_native_prepare'] and CFG['training_mode']=='gpu_neuroevolution':
        import urllib.request
        try:
            header=MODELS/'chess.hpp'
            if not header.exists():
                urllib.request.urlretrieve('https://raw.githubusercontent.com/Disservin/chess-library/2a608d58f44776909bb6e2c09505e97ba1451319/include/chess.hpp',header)
            assert hashlib.sha256(header.read_bytes()).hexdigest()=='905b95c2ac870e488a5262c992e3aed0907a2233eb95b9a2835a3d47d9a3cf5d'
            # The pinned header includes the complete MIT license.
            source_path=MODELS/'cristopher_gpu_children.cpp';source_path.write_text(GPU_NATIVE_SOURCE)
            binary=MODELS/'cristopher_gpu_children.so'
            subprocess.check_call(['g++','-std=c++17','-O3','-DNDEBUG','-shared','-fPIC',str(source_path),'-o',str(binary)])
            for i,board in enumerate(positions[:8]):
                item=(i,board.fen(),[])
                a=gpu_worker.gpu_prepare_one(item);b=gpu_worker.gpu_prepare_one(item,str(binary))
                assert a[1]==b[1] and np.array_equal(a[2],b[2]) and np.array_equal(a[3],b[3],equal_nan=True)
            CFG['gpu_native_path']=str(binary)
            print('Native CPU move preparation enabled; GPU neural batches ready.')
        except Exception as error:
            print('Using Python move preparation with CUDA neural batches:',error)
    print('GPU run:',CFG['run_name'],'| CPU workers:',CFG['workers'],'| parallel games per wave:',CFG['gpu_parallel_games'])


PASS: JS baseline returned legal moves for 200 randomly reached positions.
Static evaluations/sec: 11213
PASS: symmetry, mate and stalemate.
Skipping the classic CPU search benchmark; GPU training uses batched successor evaluation.
Python search enabled.
Spawn-safe worker module: /content/drive/MyDrive/chess_30m/models/cristopher_evo_worker.py
GPU run: fresh_30m_v1 | CPU workers: 2 | parallel games per wave: 512


## Saved while training
Full-pipeline checkpoints, private training branches, datasets, PGNs and validation logs live under `MyDrive/chess_30m/experiments/<run_name>/`. No old checkpoints are imported; all training artifacts belong to this new experiment.

Dense evolution saves about every 60 seconds after the current move, at generation boundaries and recoverable exit. Parameter evolution checkpoints its state and matches; sparse training saves optimizer/RNG state and the best validation network each epoch. Pipeline stage checkpoints keep the last three versions. Independent matches save after each move and resume.

Only the current incumbent or a challenger that passes independent paired validation is published to `models/latest_champion.json`. The full-engine snapshot records its evaluator, classical weights, search switches and required neural weights. Training fitness or lower neural prediction loss alone cannot replace the champion.

A hard runtime kill can lose work after the last completed Drive write. Run all with the same account and settings to recover. Keep one writer per run.


In [4]:
#@title 4 · Train and continuously save the champion { display-mode: "form" }
if CFG['training_mode']=='full_pipeline':
    STATE=gpu_worker._CHESS_ENGINE_PACKAGE.full_pipeline.run(gpu_worker,CFG)
    print('Pipeline cycle:',STATE['cycle'],'| next stage:',gpu_worker._CHESS_ENGINE_PACKAGE.full_pipeline.STAGES[STATE['stage']])
elif CFG['training_mode']=='gpu_neuroevolution':
    STATE=gpu_worker.gpu_evolve(CFG)
    print('Completed GPU generations:',STATE['generation'], '| selected champion generation:',STATE['champion_generation'])
elif CFG['training_mode']=='parameter_evolution':
    STATE=gpu_worker._CHESS_ENGINE_PACKAGE.notebook_bridge.run_parameters(gpu_worker,CFG)
    print('Completed parameter generations:',STATE['generation'])
else:
    STATE=worker.evolve(CFG)
    print('Completed classic generations:',STATE['generation'])


Champion saved to Drive | generation -1 | 3903358b4c986a95-6207654a
Full pipeline: cycle 0, stage dense, 150.0 minutes left
Training GPU: Tesla T4
GPU resume: gpu_00000001.pt generation 0
Neural population device: cuda | 35 networks | 30,017,409 parameters each
Generation 0 | 792 paired games | CPU workers 1 | CUDA allocated 3.97 GiB, reserved 11.76 GiB, peak 3.97 GiB
Search-based ES: generation 0, game 6, ply 5
Search-based ES: generation 0, game 10, ply 64
Search-based ES: generation 0, game 12, ply 96
Search-based ES: generation 0, game 16, ply 35
Search-based ES: generation 0, game 17, ply 157
Search-based ES: generation 0, game 18, ply 217
Search-based ES: generation 0, game 19, ply 83
Search-based ES: generation 0, game 26, ply 35
Search-based ES: generation 0, game 37, ply 17
Search-based ES: generation 0, game 41, ply 31
Search-based ES: generation 0, game 46, ply 19
Search-based ES: generation 0, game 52, ply 55
Search-based ES: generation 0, game 56, ply 51
Search-based ES: g

: 

: 

In [ ]:
#@title 5 · Verify the saved champion { display-mode: "form" }
# Checkpoints and deployment snapshots are already saved throughout GPU training.
if CFG['training_mode']=='full_pipeline':
    STATE=gpu_worker._CHESS_ENGINE_PACKAGE.full_pipeline.load(gpu_worker,CFG)
    EXPORTED_MODELS=gpu_worker._CHESS_ENGINE_PACKAGE.full_pipeline.publish(gpu_worker,CFG,STATE)
    print('Selected deployment evaluator:',STATE['champion']['kind'],'| verified:',STATE['verified'])
    print('Full pipeline progress is saved; Run all resumes the next incomplete stage.')
elif CFG['training_mode']=='gpu_neuroevolution':
    STATE=gpu_worker.gpu_load(CFG)
    EXPORTED_MODELS=gpu_worker.gpu_export(CFG,STATE,Path(gpu_worker.__file__).read_text())
    print('Selected champion generation:',STATE['champion_generation'])
    if STATE['champion_generation'] < 0:
        print('No candidate has passed the promotion gate yet; the saved champion is the initial baseline.')
elif CFG['training_mode']=='parameter_evolution':
    STATE=gpu_worker._CHESS_ENGINE_PACKAGE.notebook_bridge.load_parameters(CFG)
    EXPORTED_MODELS=gpu_worker._CHESS_ENGINE_PACKAGE.notebook_bridge.parameter_export(gpu_worker,CFG,STATE)
else:
    STATE=worker.load_state(CFG)
    EXPORTED_MODELS=worker.publish_classic_champion(CFG,STATE,Path(worker.__file__).read_text())
    if EXPORTED_MODELS is None:
        print('Classic training has not completed a generation yet. Run all again to resume its saved games.')
if EXPORTED_MODELS is not None:
    print('Deployment-ready champion:',EXPORTED_MODELS)
    print('Next: open deploy.ipynb in Colab and choose Run all.')

if EXPORTED_MODELS is not None:
    fresh_test = r"""
    import importlib.util, json, sys
    from pathlib import Path
    import chess
    folder=Path(sys.argv[1])
    spec=importlib.util.spec_from_file_location('fresh_cristopher',folder/'cristopher_engine.py')
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    engine=module.CristopherEngine(folder/'cristopher_genome.json')
    fens=[chess.STARTING_FEN,'r3k2r/8/8/8/8/8/8/R3K2R b KQkq - 0 1',
          '4k3/8/8/3pP3/8/8/8/4K3 w - d6 0 1','1r5k/P7/8/8/8/8/8/7K w - - 0 1',
          '7k/8/5KQ1/8/8/8/8/8 w - - 0 1']
    for i,fen in enumerate(fens):
        answer=engine.get_move(fen,None,('easy','medium','hard')[i%3])
        assert chess.Move.from_uci(answer['move']) in chess.Board(fen).legal_moves
        assert isinstance(answer['info']['eval'],float)
        assert all(isinstance(answer['info'][key],int) for key in ('depth','nodes','time_ms'))
        print(json.dumps(answer))
    print('PASS: five FENs loaded from the exported Drive files in a fresh process.')
    """
    import textwrap
    fresh_test=textwrap.dedent(fresh_test)
    (ROOT/'logs'/CFG['run_name']).mkdir(parents=True,exist_ok=True)
    verified=subprocess.run([sys.executable,'-c',fresh_test,str(EXPORTED_MODELS)],capture_output=True,text=True,check=True,timeout=60)
    print('Export check: five legal moves from a fresh Python process.')
    (ROOT/'logs'/CFG['run_name']/'export_check.txt').write_text(verified.stdout)

# Optional independent comparison using the ACTUAL Drive champion on this runtime.
VALIDATE_SAVED_CHAMPION = False #@param {type:'boolean'}
VALIDATION_PAIRS = 32 #@param {type:'integer'}
VALIDATION_SECONDS_PER_MOVE = 0.5 #@param {type:'number'}
VALIDATION_HOURS = 3.0 #@param {type:'number'}
if VALIDATE_SAVED_CHAMPION:
    if CFG['training_mode'] != 'gpu_neuroevolution':
        raise ValueError('Full pipeline already validates search changes automatically; use this extra legacy comparison only in gpu_neuroevolution mode.')
    if VALIDATION_PAIRS < 32 or VALIDATION_SECONDS_PER_MOVE <= 0 or VALIDATION_HOURS <= 0:
        raise ValueError('Use at least 32 pairs and positive time budgets.')
    SAVED_CHAMPION_VALIDATION = gpu_worker._CHESS_ENGINE_PACKAGE.notebook_bridge.validate_saved_dense_search(
        gpu_worker, CFG, EXPORTED_MODELS, VALIDATION_PAIRS, VALIDATION_SECONDS_PER_MOVE, VALIDATION_HOURS)
    print(json.dumps(SAVED_CHAMPION_VALIDATION, indent=2))


: 

## Next: deploy
Open the updated **[deploy.ipynb](deploy.ipynb)** in a separate Colab runtime and choose **Run all**. It loads the selected full-engine bundle, including its evaluator and validated search configuration, and prints the tested API URL.

Run all in this training notebook again to continue the next incomplete pipeline stage. A 2.5-hour session does not guarantee completion of a full validation cycle. You can change TRAINING_HOURS while resuming; changing experiment settings requires a new RUN_NAME. Saved progress belongs only to this new experiment.

[Engine and training guide](../docs/ENGINE_GUIDE.md). The optional `VALIDATE_SAVED_CHAMPION` switch in section 5 is for the legacy dense-only mode; full_pipeline already schedules independent comparisons.
